# Apache HBase

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Apache HBase is a distributed, scalable, wide-column NoSQL store modeled on Google Bigtable and built on top of HDFS. It gives random, real-time reads and writes by row key over tables with billions of rows and millions of columns, while HDFS provides the durable storage underneath.

```text
client -> ZooKeeper (who is the master, where is hbase:meta)
       -> RegionServer -> region -> MemStore + WAL + HFiles on HDFS
```

Key ideas:

+ Rows are sorted lexicographically by row key and split into regions; a region is the unit of distribution, not a table.
+ A column family is the physical storage and tuning unit; column qualifiers inside it are fully dynamic.
+ Writes land in the write-ahead log (WAL) and the in-memory MemStore, and are flushed to immutable HFiles later.
+ Reads check the MemStore and block cache, then the HFiles; tombstones hide deletes until compaction removes them.
+ HBase is optimized for key-based access and scans, not for joins, ad-hoc SQL analytics, or multi-row transactions across tables.

## Contents

<details>
<summary>56 sections - click to expand</summary>

- [How do you design an HBase row key?](#how-do-you-design-an-hbase-row-key)
- [When should you use HBase?](#when-should-you-use-hbase)
- [How do you create and query tables in the HBase shell?](#how-do-you-create-and-query-tables-in-the-hbase-shell)
- [How do you configure an HBase cluster with hbase-site.xml?](#how-do-you-configure-an-hbase-cluster-with-hbase-sitexml)
- [How do you read and write data with the HBase Java client?](#how-do-you-read-and-write-data-with-the-hbase-java-client)
- [How do you use HBase from Python with HappyBase?](#how-do-you-use-hbase-from-python-with-happybase)
- [How do you query HBase with Apache Phoenix SQL?](#how-do-you-query-hbase-with-apache-phoenix-sql)
- [What are the differences between RDBMS and HBase data model?](#what-are-the-differences-between-rdbms-and-hbase-data-model)
- [Mention what are the key components of HBase?](#mention-what-are-the-key-components-of-hbase)
- [Mention how many operational commands in HBase?](#mention-how-many-operational-commands-in-hbase)
- [Explain what is WAL and HLog in HBase?](#explain-what-is-wal-and-hlog-in-hbase)
- [In HBase what is column families?](#in-hbase-what-is-column-families)
- [Explain deletion in HBase? Mention what are the three types of tombstone markers in HBase?](#explain-deletion-in-hbase-mention-what-are-the-three-types-of-tombstone-markers-in-hbase)
- [Explain how does HBase actually delete a row?](#explain-how-does-hbase-actually-delete-a-row)
- [Explain what happens if you alter the block size of a column family on an already occupied database?](#explain-what-happens-if-you-alter-the-block-size-of-a-column-family-on-an-already-occupied-database)
- [What is HBaseFsck class?](#what-is-hbasefsck-class)
- [What are the main key structures of HBase?](#what-are-the-main-key-structures-of-hbase)
- [Discuss how you can use filters in Apache HBase](#discuss-how-you-can-use-filters-in-apache-hbase)
- [What is the meaning of compaction in HBase?](#what-is-the-meaning-of-compaction-in-hbase)
- [How will you implement joins in HBase?](#how-will-you-implement-joins-in-hbase)
- [What is the use of MasterServer?](#what-is-the-use-of-masterserver)
- [What is a cell in HBase?](#what-is-a-cell-in-hbase)
- [What is a Bloom filter?](#what-is-a-bloom-filter)
- [Which technique can you use in HBase to access HFile directly without the help of HBase?](#which-technique-can-you-use-in-hbase-to-access-hfile-directly-without-the-help-of-hbase)
- [Can the region server will be located on all DataNodes?](#can-the-region-server-will-be-located-on-all-datanodes)
- [Define the difference between Hive and HBase?](#define-the-difference-between-hive-and-hbase)
- [Explain the data model of HBase.](#explain-the-data-model-of-hbase)
- [What is RegionServer?](#what-is-regionserver)
- [What are the data manipulation commands of HBase?](#what-are-the-data-manipulation-commands-of-hbase)
- [What is the use of ZooKeeper?](#what-is-the-use-of-zookeeper)
- [What are the step in writing something into HBase by a Client?](#what-are-the-step-in-writing-something-into-hbase-by-a-client)
- [What are the different compaction types in HBase?](#what-are-the-different-compaction-types-in-hbase)
- [What is the Scope of a Rowkey in HBase?](#what-is-the-scope-of-a-rowkey-in-hbase)
- [What is a Namespace in HBase?](#what-is-a-namespace-in-hbase)
- [What is Hotspotting in HBase?](#what-is-hotspotting-in-hbase)
- [What is Ttl in HBase?](#what-is-ttl-in-hbase)
- [Why do we Pre create empty Regions?](#why-do-we-pre-create-empty-regions)
- [What are the Two Types of Table Design Approach in HBase?](#what-are-the-two-types-of-table-design-approach-in-hbase)
- [When do we do Manual Region Splitting?](#when-do-we-do-manual-region-splitting)
- [In HBase what is Log Splitting?](#in-hbase-what-is-log-splitting)
- [How does HBase support Bulk Data Loading?](#how-does-hbase-support-bulk-data-loading)
- [Why MultiWAL is Needed?](#why-multiwal-is-needed)
- [How does HBase provide High Availability?](#how-does-hbase-provide-high-availability)
- [How does Wal help when a Regionserver Crashes?](#how-does-wal-help-when-a-regionserver-crashes)
- [What are the different Block Caches in HBase?](#what-are-the-different-block-caches-in-hbase)
- [How do you manage compactions and region splits?](#how-do-you-manage-compactions-and-region-splits)
- [When should you not use HBase, and what should you use instead?](#when-should-you-not-use-hbase-and-what-should-you-use-instead)
- [How do you design a row key for even distribution and diagnose a hotspot?](#how-do-you-design-a-row-key-for-even-distribution-and-diagnose-a-hotspot)
- [How do you size regions, pre-split a table, and choose a split policy?](#how-do-you-size-regions-pre-split-a-table-and-choose-a-split-policy)
- [How do you tune the read path?](#how-do-you-tune-the-read-path)
- [How do you tune the write path?](#how-do-you-tune-the-write-path)
- [Things to remember: key numbers and limits](#things-to-remember-key-numbers-and-limits)
- [Things to remember: semantics to memorize](#things-to-remember-semantics-to-memorize)
- [Things to remember: mistakes that keep coming back](#things-to-remember-mistakes-that-keep-coming-back)
- [Real world use cases for HBase](#real-world-use-cases-for-hbase)
- [HBase exercises](#hbase-exercises)

</details>

Tip: the links open the sections on GitHub; in Jupyter or Colab, use the notebook's outline panel in the sidebar instead.

## How do you design an HBase row key?

An HBase row key should support the main lookup pattern and avoid region hotspotting. Because rows are sorted by key and a region serves a contiguous key range, monotonically increasing keys (timestamps, sequential ids) overload the last region. Use salting, hashing, bucketing, or reversed timestamps to spread writes.

Common patterns:

| Access pattern | Row key design |
|---|---|
| latest events for a user | `user_id + reverse_timestamp` |
| devices by tenant | `salt + tenant_id + device_id` |
| time-range scans, spread writes | `salt + reverse_timestamp + id` |
| lookup by many secondary keys | separate index/lookup table keyed by the secondary key |
| natural unique id | hash(id) or a fixed-width, zero-padded id |

```text
raw row key:        device-0001|2026-10-04T13:20:00Z
salted row key:     07|device-0001|2026-10-04T13:20:00Z   (salt = hash(device_id) % 16)
bucketed row key:   device-0001|bkt-3|1700000000000
reversed timestamp: device-0001|9999999999999-1700000000000
```

In [ ]:
# Build a salted, reverse-timestamp row key for recent-first lookups
import hashlib

def rowkey(device_id: str, ts_ms: int, buckets: int = 16) -> str:
    salt = int(hashlib.md5(device_id.encode()).hexdigest(), 16) % buckets
    reverse_ts = 9_999_999_999_999 - ts_ms
    return f"{salt:02d}|{device_id}|{reverse_ts:013d}"

The trade-off: reads for one device must query every salt bucket (or you scan all prefixes) unless the salt is derivable from the query. Salting spreads writes but fans out reads; reverse timestamps give newest-first scans for a known prefix.

**Mechanics.** HBase routes by key range, not by row identity. `hbase:meta` is itself an HBase table whose row keys are `table,startKey,regionId`; the client scans it once, caches the region boundaries, and binary-searches them for every request. The row key therefore decides both the sort order inside a region and which RegionServer answers. A key that increases forever (a raw timestamp, a sequential id) always sorts past every other key, so it always lands in the last region and one machine absorbs every write.

**Worked example.** A raw event key `device-0001|2026-10-04T13:20:00Z` puts all traffic for one device in one region. Rewritten as `07|device-0001|9999999999999-1700000000000`, the device moves into bucket 7 of 16; the bucket prefixes `00` to `0f` are spread across the table's regions, and the reversed timestamp sorts the newest event first inside the device.

**Common mistake.** Salting with two or four buckets only relocates the hotspot. Choose the bucket count so `expected rows / buckets` is close to the target region size, and format the salt at fixed width (`%02d`) so lexicographic order matches numeric order. Salting also fans out reads: a query for one device must know its bucket, or it scans all buckets and multiplies read load.

## When should you use HBase?

Use HBase for sparse, wide, high-scale tables that need random reads/writes by key over large datasets. It fits operational lookups, time-series style access, and serving stores behind an application. It is not ideal for ad-hoc SQL analytics, joins, or full table scans.

Example:

+ good: get the latest 100 events for a device id
+ poor: calculate total revenue by region across all history

Analytics usually belongs in Hive, Spark, Impala, or a warehouse, while HBase serves key-based access. A common pattern is to serve from HBase and run analytics on the same data copied to Parquet or Iceberg.

**Mechanics.** HBase keeps one copy of the data sorted by row key and serves it from RegionServers that read local HDFS blocks, so it is strong at `get` by key and bounded range scans, and weak at aggregation, joins, and predicate scans that touch most of the table. Decide from the access pattern, not from the data volume: a terabyte queried one key at a time is an HBase problem, a terabyte aggregated by region is not.

**Worked example.** "Latest 100 readings for device 42" becomes a single region lookup with a `LIMIT` once the key is `salt|device|reverse_ts`, and returns in single-digit milliseconds. "Total revenue by region for the last year" must read every row in the table, because there is no secondary index and no columnar layout.

**Common mistake.** Treating HBase as a drop-in RDBMS replacement. Ports that keep SQL-shaped access patterns (secondary lookups, joins, `WHERE` on a non-key column) turn into full scans and fail at scale. The usual fix is to serve from HBase and copy the same data to Parquet or Iceberg for analytics.

## How do you create and query tables in the HBase shell?

The HBase shell is a JRuby REPL that talks to the cluster for DDL and DML. Namespaces, tables, and column families are created here; `disable` is required before changing a table.

```bash
hbase shell
```

```bash
# Namespace and table DDL: TTL, versions, block size, and Bloom filter per column family
create_namespace 'app'
create 'app:impressions', {NAME => 'i', VERSIONS => 1, TTL => 7776000, BLOOMFILTER => 'ROWCOL', BLOCKSIZE => 65536}
describe 'app:impressions'

# Add a column family to an existing table
disable 'app:impressions'
alter 'app:impressions', {NAME => 'meta', VERSIONS => 3}
enable 'app:impressions'
```

```bash
# DML: put, get, scan, increment, delete, count
put 'app:impressions', 'u42|9999999999999-1700000000000|a1', 'i:campaign_id', 'cmp-99'
put 'app:impressions', 'u42|9999999999999-1700000000000|a1', 'i:amount', '42.10'
get 'app:impressions', 'u42|9999999999999-1700000000000|a1'
scan 'app:impressions', {STARTROW => 'u42|', ENDROW => 'u42|\xff', LIMIT => 5}
incr 'app:counters', 'daily|2026-10-04', 'c:impressions', 1
delete 'app:impressions', 'u42|9999999999999-1700000000000|a1', 'i:amount'
count 'app:impressions'
```

```bash
# Table maintenance and reporting
status 'detailed'
list_regions 'app:impressions'
flush 'app:impressions'
major_compact 'app:impressions'
```

**Mechanics.** The shell is a JRuby REPL that wraps the Java admin and table APIs, so every command maps to a real client call: `create` builds a table in `hbase:meta` and assigns regions, `put` and `get` go through the RegionServer, and `disable` blocks traffic while the master edits the schema. `list` reads the master's table state, not the data.

**Worked example.** A create with explicit families, a put, and a bounded scan is the whole loop:

```bash
create 'app:impressions', {NAME => 'i', VERSIONS => 1, TTL => 7776000}, SPLITS => ['10','20','30']
put 'app:impressions', '07|u42|1', 'i:campaign_id', 'cmp-99'
get 'app:impressions', '07|u42|1'
scan 'app:impressions', {STARTROW => '07|u42|', ENDROW => '07|u42|ÿ', LIMIT => 10}
```

**Common mistake.** Using an unbounded `scan` in the shell on a production table. With no `STARTROW`, `ENDROW`, `LIMIT`, or filter it streams the whole table through one client and can pin a RegionServer for minutes. Always bound a shell scan.

## How do you configure an HBase cluster with hbase-site.xml?

`hbase-site.xml` holds cluster-wide settings; the most important ones control durability, write throughput, and MemStore sizing. The pattern is one `property` block per setting.

```xml
<configuration>
  <property>
    <name>hbase.rootdir</name>
    <value>hdfs://nn.example.com:8020/hbase</value>
  </property>
  <property>
    <name>hbase.cluster.distributed</name>
    <value>true</value>
  </property>
  <property>
    <name>hbase.zookeeper.quorum</name>
    <value>zk1.example.com,zk2.example.com,zk3.example.com</value>
  </property>
  <property>
    <name>hbase.regionserver.handler.count</name>
    <value>60</value>
  </property>
  <property>
    <name>hbase.regionserver.global.memstore.size</name>
    <value>0.4</value>
  </property>
  <property>
    <name>hbase.hregion.memstore.flush.size</name>
    <value>134217728</value>
  </property>
  <property>
    <name>hbase.hregion.max.filesize</name>
    <value>10737418240</value>
  </property>
  <property>
    <name>hbase.hstore.blockingStoreFiles</name>
    <value>16</value>
  </property>
  <property>
    <name>hbase.wal.provider</name>
    <value>filesystem</value>
  </property>
  <property>
    <name>hbase.hregion.memstore.mslab.enabled</name>
    <value>true</value>
  </property>
</configuration>
```

Tuning notes:

+ `hbase.regionserver.global.memstore.size = 0.4` caps total MemStore memory at 40% of heap; the rest serves the block cache and reads.
+ Lower `hbase.hregion.memstore.flush.size` when write bursts are spiky; larger values reduce flush frequency but increase heap pressure.
+ `hbase.hregion.max.filesize` controls when a region splits; 10 GB is a common production value.
+ `hbase.hstore.blockingStoreFiles` makes the RegionServer block writes when too many store files pile up, which pushes back on the client instead of failing.

**Mechanics.** `hbase-site.xml` is loaded on every daemon and client, and the master pushes some values into `hbase:meta` at table creation time, so a few settings are written once and stay with the table. The three memory dials must add up: `hbase.regionserver.global.memstore.size` caps total MemStore use, `hfile.block.cache.size` caps the read cache, and their sum must stay under the heap so the JVM has room for request buffers and the WAL.

**Worked example.** On a 32 GB RegionServer heap, `hbase.regionserver.global.memstore.size = 0.4` allows about 12.8 GB of MemStore, `hfile.block.cache.size = 0.2` gives about 6.4 GB of cache, and the remaining 40 percent covers handlers, MSLAB overhead, and GC headroom. With `hbase.hregion.memstore.flush.size = 134217728` (128 MB), that MemStore budget holds roughly 100 region MemStores before global flushing forces pressure.

**Common mistake.** Copying a `hbase-site.xml` between clusters of different heap sizes. The fractions are relative to the heap, so the same file means a different number of bytes, and on a smaller heap the combined fractions leave no GC headroom and the RegionServer dies under load.

## How do you read and write data with the HBase Java client?

The Java client is the canonical API. Create the `Connection` once per process and reuse it; `Table` and `Admin` are cheap to obtain from it. Batch and async paths are used for high throughput.

```java
Configuration conf = HBaseConfiguration.create();
conf.set("hbase.zookeeper.quorum", "zk1,zk2,zk3");
```


Rules that matter under load: never build a `Connection` per request, set `Scan.setCaching` (rows per RPC) and `setBatch` (cells per RPC), keep row keys short, and prefer `BufferedMutator` over many single `put` calls.

**Mechanics.** The Java client is the canonical path: it resolves region locations from `hbase:meta`, caches them, and sends `Mutate` and `Scan` RPCs straight to the RegionServer. `Connection` owns the RPC pool and the meta cache, so it must be created once per process; `Table` and `Admin` are lightweight handles. Under load, `BufferedMutator` batches writes per region, and `Scan.setCaching` controls rows per RPC while `setBatch` controls cells per RPC.

```java
Configuration conf = HBaseConfiguration.create();
try (Connection conn = ConnectionFactory.createConnection(conf);
     Table table = conn.getTable(TableName.valueOf("app:impressions"))) {
    Get get = new Get(Bytes.toBytes("07|u42|1")).addColumn(Bytes.toBytes("i"), Bytes.toBytes("campaign_id"));
    Result r = table.get(get);
    Scan scan = new Scan().withStartRow(Bytes.toBytes("07|u42|")).setCaching(200).setBatch(100);
}
```

**Common mistake.** Creating a `Connection` per request. Each connection starts its own executor, meta cache, and RPC threads; a few thousand per second exhaust sockets and heap. Also avoid a `Scanner` left open: it holds a server-side lease and blocks the region.

## How do you use HBase from Python with HappyBase?

HappyBase talks to the Thrift gateway (`hbase-daemon.sh start thrift`). It is convenient for services and scripts, but the Thrift gateway is an extra hop compared with the native Java API.

In [ ]:
# ILLUSTRATIVE - requires happybase: install to run, or read on.
import happybase

connection = happybase.Connection("hbase-thrift.example.com", port=9090)
table = connection.table("app:impressions")

# Put a row
table.put(
    b"u42|9999999999999-1700000000000|a1",
    {b"i:campaign_id": b"cmp-99", b"i:amount": b"42.10"},
)

# Point read
row = table.row(b"u42|9999999999999-1700000000000|a1")
print(row.get(b"i:campaign_id"))

# Prefix scan for the most recent impressions of a user
for key, data in table.scan(row_prefix=b"u42|", limit=10):
    print(key, data)

# Atomic counter
counters = connection.table("app:counters")
counters.counter_inc(b"daily|2026-10-04", b"c:impressions", 1)

connection.close()

Use `happybase.ConnectionPool` in a web service so connections are reused, and keep the Thrift server close to the RegionServers to avoid an extra network round trip per scan.

**Mechanics.** HappyBase speaks the Thrift gateway, which runs as a separate daemon next to the RegionServers and translates each call into the Java client. It is a real client, not a driver: connection setup is expensive, `ConnectionPool` is the thread-safe wrapper, and filters are passed as strings that the gateway parses.

**Worked example.** A pooled connection reuses the Thrift socket per thread and a bounded prefix scan fetches one user's recent rows:

```python
import happybase
pool = happybase.ConnectionPool(size=4, host="hbase-thrift.example.com", port=9090)
with pool.connection() as conn:
    for key, data in conn.table("app:impressions").scan(row_prefix=b"07|u42|", limit=10):
        print(key, data)
```

**Common mistake.** Assuming HappyBase sees uncommitted MemStore state instantly across regions, or that a scan is cheap because Python is slow anyway. The Thrift hop adds latency per RPC, so batch by `batch_size` and keep the gateway on the same rack as the RegionServers. `happybase` is not installed in this repository, so the cell above is review-only.

## How do you query HBase with Apache Phoenix SQL?

Phoenix adds a SQL layer over HBase: it compiles SQL to HBase scans, maintains secondary indexes in HBase tables, and gives JDBC access. The row key is still the primary key, and the primary key order defines the physical sort order.

```sql
-- Primary key maps to the HBase row key; the leading columns drive the scan
CREATE TABLE IF NOT EXISTS impressions (
    user_id   VARCHAR NOT NULL,
    ts        TIMESTAMP NOT NULL,
    campaign  VARCHAR,
    amount    DECIMAL(10,2),
    CONSTRAINT pk PRIMARY KEY (user_id, ts DESC)
);

UPSERT INTO impressions (user_id, ts, campaign, amount)
VALUES ('u42', CURRENT_TIME(), 'cmp-99', 42.10);

-- Secondary index for queries that do not lead with the primary key
CREATE INDEX impressions_campaign_idx ON impressions (campaign);

SELECT campaign, COUNT(*), SUM(amount)
FROM impressions
WHERE user_id = 'u42'
GROUP BY campaign;
```

Phoenix is the right answer when interviewers ask "does HBase support SQL?" - **not natively, but Phoenix provides it**. Beware that a Phoenix query that does not lead with the primary key may fall back to a full scan unless a secondary index exists.

**Mechanics.** Phoenix compiles SQL into HBase `Scan` and `Get` operations and stores rows as HBase cells, with the primary key columns concatenated into the row key in declared order. A query is fast only when its `WHERE` clause constrains a leading prefix of that key, because only then does Phoenix produce a bounded scan; otherwise it reads every row and filters client-side. Secondary indexes are separate HBase tables maintained by Phoenix on write (or asynchronously with `ASYNC`).

**Worked example.** With `PRIMARY KEY (user_id, ts DESC)`, `WHERE user_id = 'u42' AND ts > ...` is a bounded scan, and `ORDER BY ts DESC LIMIT 10` is free because the key is already reversed. `WHERE campaign = 'cmp-99'` has no leading-key constraint and needs `CREATE INDEX ... ON impressions (campaign)` or it becomes a full scan.

**Common mistake.** Writing an index for every column. Each global index is another HBase table updated on every write, so write throughput falls and the index tables need their own tuning. Index the queries that matter, and check `EXPLAIN` to confirm a range scan rather than `FULL SCAN`.

## What are the differences between RDBMS and HBase data model?

Schema: RDBMS defines tables and typed columns up front. HBase defines tables and column families, but column qualifiers inside a family are dynamic and untyped (everything is bytes), so the schema is much looser.

Normalization: an RDBMS is normalized; HBase is deliberately denormalized and stores wide, sparse rows.

Partitioning: an RDBMS requires manual partitioning and sharding. HBase splits a table into regions automatically as it grows, and can be pre-split manually.

Other differences: RDBMS supports joins, multi-row transactions, and SQL; HBase supports single-row atomicity, no joins, and no server-side SQL without Phoenix. HBase tables are sparse - a row only stores the cells that exist.

**Mechanics.** The difference is where the schema lives. An RDBMS fixes typed columns in a catalogue and enforces them on every write. HBase fixes only tables and column families in `hbase:meta`; qualifiers inside a family are created on first write, and every value is an uninterpreted byte array, so the schema is enforced by the application or by Phoenix.

**Worked example.** A product row `sku-99321` can carry `p:color`, `p:voltage`, and `p:size` for three different categories in the same table. Adding `p:weight` needs no `ALTER TABLE`; a relational equivalent needs either a wide table full of `NULL`s or an entity-attribute-value table joined back on every read.

**Common mistake.** Reading "column-family oriented" as "columnar". HBase stores all cells of a family for one row together, which suits row-at-a-time serving; Parquet stores each column separately, which suits analytics. They optimize opposite access patterns, so HBase is not a substitute for a columnar warehouse.

## Mention what are the key components of HBase?

+ ZooKeeper: coordinates the cluster, holds the `hbase:meta` location and master election.
+ HBase Master (HMaster): assigns regions, balances load, and performs admin operations.
+ RegionServer: serves regions and handles reads, writes, and flushes.
+ Region: a contiguous range of a table's rows, containing a MemStore and HFiles.
+ Catalog tables: `hbase:meta` maps regions to RegionServers (the older `-ROOT-` table is gone in modern HBase).
+ HDFS: durable storage for WALs and HFiles.

**Mechanics.** The components form a strict hierarchy. ZooKeeper holds the `hbase:meta` location and elects the active master. The HMaster assigns regions and balances load but is off the data path. Each RegionServer hosts regions, appends to the WAL, keeps MemStores, and flushes HFiles to HDFS. `hbase:meta` is the catalogue that maps a key range to the region and server that owns it.

```text
ZooKeeper -> HMaster (assignment, DDL, balancing)
          -> hbase:meta (key range -> region -> RegionServer)
          -> RegionServer -> region -> store (one per family) -> MemStore + HFiles on HDFS
```

**Common mistake.** Believing the client asks the master where data lives. It does not: clients cache region locations from `hbase:meta` and talk to RegionServers directly, which is why a master restart does not interrupt serving and why a stale client cache shows up as `RegionMovedException` retries.

## Mention how many operational commands in HBase?

The core data operations are five: Get, Put, Delete, Scan, and Increment. Around them sit admin operations such as Create, Disable/Enable, Alter, Drop, Truncate, Split, Merge, Flush, and Compact.

```bash
# One of each core operation in the shell
put 'app:impressions', 'u42|1', 'i:campaign', 'cmp-99'
get 'app:impressions', 'u42|1'
scan 'app:impressions', {LIMIT => 10}
incr 'app:counters', 'daily|2026-10-04', 'c:impressions', 1
delete 'app:impressions', 'u42|1', 'i:campaign'
```

**Mechanics.** The count depends on how you slice it. The five core data operations are Get, Put, Delete, Scan, and Increment, with Append as a sixth cell-level operation. Around them sit table operations (Create, Alter, Disable, Enable, Drop, Truncate), region operations (Split, Merge, Assign, Move), and maintenance operations (Flush, Compact, Major compact, Balance).

```bash
# Core operations
put 'app:impressions', 'u42|1', 'i:campaign', 'cmp-99'
get 'app:impressions', 'u42|1'
scan 'app:impressions', {LIMIT => 10}
incr 'app:counters', 'daily|2026-10-04', 'c:impressions', 1
delete 'app:impressions', 'u42|1', 'i:campaign'
# Maintenance
flush 'app:impressions'
compact 'app:impressions'
major_compact 'app:impressions'
```

**Common mistake.** Answering with a number and stopping. The interviewer wants the taxonomy plus the distinction that `increment` is atomic and server-side while a read-modify-write `put` of a counter is not, and that `truncate` is a disable, drop, and recreate, which destroys pre-splits and TTL settings you configured.

## Explain what is WAL and HLog in HBase?

The WAL (Write-Ahead Log), historically called HLog, records every mutation before it is applied to memory. It is a Hadoop sequence file of `WALKey` plus cells, and it is used to replay not-yet-flushed data after a server crash. On RegionServer failure, the master splits the WAL by region and replays each region's edits so committed writes are not lost.

```properties
# Durability settings in hbase-site.xml
hbase.wal.provider = filesystem
# ASYNC_WAL is faster but can lose recent writes on crash; SYNC_WAL is the default
```

**Mechanics.** A mutation is appended to the WAL before it reaches the MemStore, so an acknowledged write survives a crash. The WAL is a rolled sequence of HDFS files; each entry carries a `WALKey` (region, table, sequence id) and the cells. When the MemStore is flushed, the WAL sequence number for that region advances and old WAL files can be archived or deleted. `Durability` per mutation controls whether the append is synced to disk.

**Worked example.** A RegionServer dies with 4 GB of unflushed MemStore. The master splits the dead server's WAL by region, and each region's edits are replayed into a new MemStore on the reassigned RegionServer. Committed writes reappear; nothing is lost except writes made with `Durability.ASYNC_WAL`.

**Common mistake.** Setting `Durability.SKIP_WAL` or `ASYNC_WAL` cluster-wide to chase throughput. That trades a small write-latency win for real data loss on crash, and it is almost never the right first lever. Tune MemStore size and compaction before weakening durability.

## In HBase what is column families?

A column family is the basic unit of physical storage and the level at which storage features such as compression, block size, Bloom filters, versions, and TTL are configured. All cells in a family for a row are stored together on the same RegionServer, so families should be defined by access pattern, not by convenience.

```bash
# One family for the hot lookup path, one for cold metadata
create 'app:impressions', {NAME => 'i', VERSIONS => 1}, {NAME => 'meta', VERSIONS => 3}
```

Keep the number of column families small (typically 1-3): each family has its own MemStore and flush behavior, and many families multiply memory pressure and flush storms.

**Mechanics.** A column family is the unit of physical storage, memory, and tuning. Every region has one store per family, each store has its own MemStore and its own set of HFiles, and family settings such as `VERSIONS`, `TTL`, `COMPRESSION`, `BLOCKSIZE`, `BLOOMFILTER`, and `IN_MEMORY` live at this level. Qualifiers inside a family share all of it, so two qualifiers with different access patterns belong in different families.

**Worked example.** The hot lookup path gets one family with one version and a row Bloom filter; large cold documents get a second family with compression and a longer TTL. A read of the hot family does not touch the cold family's files, which keeps the block cache focused.

**Common mistake.** Creating a family per logical attribute "for tidiness". Each family has its own MemStore, so ten families multiply flush pressure and memory, and a `get` for a row touches ten stores. The common guidance is one to three families per table; more than that needs a specific justification.

## Explain deletion in HBase? Mention what are the three types of tombstone markers in HBase?

When you delete a cell, the data is not immediately removed; a tombstone marker is written that makes the deleted cells invisible, and the data is physically removed during compaction.

Three types of tombstone marker:

+ Version delete marker: marks a single version of a column.
+ Column delete marker: marks all versions of one column.
+ Family delete marker: marks all columns in a column family.

**Mechanics.** A delete is a write. HBase inserts a tombstone cell at the delete's timestamp, and reads discard any cell at or below it. Tombstones come in three scopes: a version marker for one version, a column marker for all versions of one qualifier, and a family marker for every qualifier in a family (used by `deleteall`). Physical removal happens only when a compaction rewrites the affected files.

**Worked example.** `delete 'app:impressions', 'u42|1', 'i:amount'` writes a column tombstone. A later `get` returns no `i:amount` even though the value bytes are still in the HFile, and the bytes disappear at the next major compaction. `count` can overstate the visible row count until tombstones are compacted, because it counts keys, not visible cells.

**Common mistake.** Deleting with an explicit old timestamp and then writing a newer value with an older timestamp. The tombstone masks the new value until a major compaction clears it. Let the server assign timestamps unless you have a specific reason not to.

## Explain how does HBase actually delete a row?

Writes go from RAM to disk as immutable HFiles (except during compaction). During deletion, a major compaction removes delete markers while minor compactions generally do not. A normal delete writes a tombstone; the data it hides is removed during compaction.

If you delete data and then insert new data with an earlier timestamp than the tombstone, subsequent Gets may still be masked by the tombstone and will not see the new value until a major compaction runs. This is why clients should write timestamps that move forward.

**Mechanics.** A delete flows through the same write path as a put: WAL append, MemStore insert, flush to an HFile containing the tombstone. Reads merge the MemStore and all HFiles, apply tombstones by timestamp, and hide matched cells. Minor compaction may merge files but keeps tombstones, because another file may still hold data the tombstone must mask. Major compaction rewrites every store file of a region, so it can safely drop tombstones and the cells they hid.

**Worked example.** A row is deleted at t=100 and rewritten at t=50. A `get` at t=101 scans the store files, sees the tombstone at 100, and suppresses the value at 50, returning nothing. After a major compaction the tombstone and the value are both gone, and a `get` correctly returns nothing as well.

**Common mistake.** Expecting disk usage to fall immediately after a delete. Space is reclaimed only at major compaction, and on a write-heavy table with TTLs and deletes, monitoring `hbase.hstore.blockingStoreFiles` and the compaction queue is what prevents a write stall.

## Explain what happens if you alter the block size of a column family on an already occupied database?

When you alter the block size of a column family, newly written data uses the new block size while old data stays in blocks of the old size. Existing data is still read correctly, and new files flushed after the change use the new block size. All data is transformed to the new block size only after the next major compaction.

**Mechanics.** Block size is a per-family attribute that was copied into each HFile's metadata when the file was written. Readers use the block size recorded in the file, so changing the family setting affects only files created afterwards. There is no in-place rewrite of existing blocks.

**Worked example.** A family created with the default 64 KB blocks is altered to `BLOCKSIZE => 16384`. The next flush writes 16 KB blocks, so new point reads decode less data per block, but old HFiles keep 64 KB blocks until they are rewritten. After a major compaction all files carry the new size.

**Common mistake.** Altering block size on a live table and expecting the read latency to change immediately. Also, altering without `disable` when the change requires it fails; some online family changes are allowed, but adding or dropping a family needs the table disabled. Measure after a major compaction, not right after the alter.

## What is HBaseFsck class?

`HBaseFsck` implements the `hbck` tool, which offers command-line switches to check region consistency and table integrity and to repair a corrupted HBase. It runs in a read-only inconsistency-identifying mode and a multi-phase read-write repair mode.

```bash
hbase hbck
hbase hbck -details
hbase hbck -repair
```

**Mechanics.** `HBaseFsck` is the class behind `hbase hbck`. It walks `hbase:meta` and HDFS, compares the region list against the actual directory layout, checks assignment consistency, and reports inconsistencies such as regions with no metadata, metadata with no directory, or holes and overlaps in the key ranges.

```bash
hbase hbck                       # read-only consistency report
hbase hbck -details              # per-region detail
hbase hbck -repair               # multi-phase repair
hbase hbck -fixMeta -fixAssignments
```

**Worked example.** After an unclean shutdown, `hbck` reports "Number of regions: 512, Deployed: 500, Not deployed: 12, holes: 0, overlaps: 0". The fix is `hbase hbck -fixAssignments`, which lets the master redeploy the twelve orphaned regions rather than hand-editing metadata.

**Common mistake.** Running `-repair` on a healthy cluster as a routine check, or running the old HBase 1.x `hbck` against HBase 2.x. In HBase 2.x consistency repair moved to the `hbck2` framework (`assigns`, `unassigns`, `fixMeta`) inside the shell. Repair is a recovery action, not maintenance.

## What are the main key structures of HBase?

The row key and the column key (column family plus qualifier) are the two most important key structures. Together with the timestamp they form the cell address: `(row key, column family, column qualifier, timestamp)`.

**Mechanics.** Two key structures matter. The row key is the physical sort key and the routing key. The column key is `family:qualifier`, and inside a store the cells for one row are ordered by family, then qualifier, then descending timestamp. Together with the timestamp they form the full cell key, so a `get` for one cell is a prefix search of that key.

```text
cell key = rowkey + family + qualifier + timestamp(descending)
row "07|u42|1" / family "i" / qualifier "amount" / newest first
```

**Worked example.** Because qualifiers are sorted, a scan with a `ColumnPrefixFilter('camp')` reads a contiguous run of the store file rather than the whole row, and a `get` with `addColumn` narrows the read to that run.

**Common mistake.** Assuming the column key has a fixed schema or that qualifiers are ordered by creation time. They are ordered byte-wise, so `amount10` sorts before `amount2`. Zero-pad numeric qualifier names when that ordering matters.

## Discuss how you can use filters in Apache HBase

Filters perform server-side filtering for scans, so rows and cells are discarded on the RegionServer before crossing the network. They were introduced in HBase 0.92 and can be used from the shell, the Java API, and Thrift. Because filtering happens after the scan reads the row from the store, filters reduce network and client work but do not eliminate the underlying I/O - a well-designed row key is still better than a filter.

```bash
# Shell examples: prefix, single-column value, and a filter list
scan 'app:impressions', {FILTER => "PrefixFilter('u42|')", LIMIT => 10}
scan 'app:impressions', {FILTER => "SingleColumnValueFilter('i', 'campaign_id', =, 'binary:cmp-99')"}
scan 'app:impressions', {FILTER => "RowFilter(>=, 'binary:u42|') AND ColumnPrefixFilter('camp')"}
scan 'app:impressions', {FILTER => "PageFilter(100)"}
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# HappyBase: a filter string is passed through to the RegionServer
for key, data in table.scan(
    row_prefix=b"u42|",
    filter=b"SingleColumnValueFilter('i', 'campaign_id', =, 'binary:cmp-99')",
    limit=50,
):
    print(key, data)

**Mechanics.** A filter is evaluated on the RegionServer inside the scan loop. `Filter.filterRowKey` runs before the row is read and can skip it cheaply; `filterCell` runs after cells are loaded and can drop them but not the I/O that read them. That split is why a `RowFilter` or `PrefixFilter` is much cheaper than a `SingleColumnValueFilter`: the first can skip a row before reading its cells, the second must read the value to decide.

**Worked example.** `PrefixFilter('u42|')` skips whole regions and rows using the key range alone. `SingleColumnValueFilter('i','campaign_id',=,'binary:cmp-99')` has to load each row's `campaign_id` cell, so it costs roughly a full scan of the key range before the filter is applied.

**Common mistake.** Using a value filter as a substitute for row key design. Filters reduce network transfer and client work, not the underlying store I/O. If a value predicate runs on every query, encode that value into the row key or build a secondary index table.

## What is the meaning of compaction in HBase?

Under heavy writes it is impractical to keep one file per store, because every read would seek across many files. Compaction merges HFiles to reduce the number of disk seeks per read and to drop deleted or expired cells. Choosing when and how aggressively to compact is a central HBase tuning topic.

**Mechanics.** Every flush creates a new immutable HFile, so the number of files per store grows with write volume, and every read must check each candidate file. Compaction merges files into fewer, larger ones and drops cells that TTL, `VERSIONS`, or tombstones have made invisible. Fewer files means fewer index lookups and disk seeks per read, at the cost of background I/O and write amplification.

**Worked example.** A store with 40 small HFiles makes a `get` consult 40 file indices and 40 Bloom filters. After compaction merges them into 2 files, the same `get` consults 2. That is the read-amplification win, and it is also why compaction competes with foreground writes for disk bandwidth.

**Common mistake.** Disabling compaction to stop the background load. Files then accumulate until `hbase.hstore.blockingStoreFiles` is crossed and the RegionServer blocks writes, which is a far worse outage than compaction I/O. Tune the compaction throughput bounds instead.

## How will you implement joins in HBase?

HBase does not support joins directly. Use a MapReduce or Spark job that scans the relevant tables and joins them, or denormalize at write time so that the data needed together is stored in one row (the preferred approach for a serving path).

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Spark can read HBase through the HBase-Spark connector and join to curated tables
impressions = (
    spark.read.format("org.apache.hadoop.hbase.spark")
    .option("hbase.table", "app:impressions")
    .option("hbase.columns.mapping", "rowkey STRING :key, campaign STRING i:campaign_id, ts STRING i:ts")
    .load()
)
campaigns = spark.table("curated.campaigns")
joined = impressions.join(campaigns, impressions.campaign == campaigns.campaign_id)

The interview point: a join in a serving path means the schema is wrong. Reshape the row key and duplicate the small side into the row instead.

**Mechanics.** There is no join operator because there is no global index or query planner over multiple tables. Two workable patterns exist: denormalize at write time so the columns needed together share a row, or run a batch job that scans both tables and performs the join in MapReduce or Spark.

**Worked example.** Instead of joining `impressions` to `campaigns` on every read, embed the campaign name in the impression row (`i:campaign_name`). The write path does one extra lookup on a small cached table, and the read path needs no join at all.

**Common mistake.** Building the join into the serving path with a per-request secondary `get`. That turns one lookup into N lookups and a network round trip for every row. If the small side is truly small, cache it in the application; if it is large, precompute the join into a wide row.

## What is the use of MasterServer?

The Master server assigns regions to RegionServers and handles load balancing across the cluster. It also performs admin operations such as create, alter, and drop table, and coordinates region splits, merges, and recovery. It is not on the read/write data path: clients talk to RegionServers directly, so a brief master outage does not stop serving.

**Mechanics.** The HMaster owns cluster metadata and coordination, not data. It runs the assignment manager, which decides which RegionServer hosts each region and rebalances when a server joins or leaves; it runs the DDL path for create, alter, drop, and truncate; and it coordinates region splits, merges, and the recovery of a failed server's WAL. ZooKeeper elects exactly one active master from the backup set.

**Worked example.** Adding a RegionServer triggers a balance: the master computes a plan and moves a fraction of regions to the new node. Killing a RegionServer triggers the opposite: its regions are reassigned and its WAL is split and replayed before those regions serve reads.

**Common mistake.** Diagnosing a client latency spike by looking at the master. The master is off the read/write path, so it is usually innocent; a stalled master shows up as schema operations hanging and regions staying unassigned, not as slow gets.

## What is a cell in HBase?

A cell is the smallest unit of an HBase table. It holds a piece of data as a tuple of `{row, column, version}`, where column is `family:qualifier` and version is the timestamp of the value.

**Mechanics.** A cell is a single `(row, family:qualifier, timestamp) -> value` entry, the smallest unit HBase reads and writes. HBase stores the raw bytes plus a type byte that marks a put, a delete, or a delete-family tombstone. Versions are ordered by timestamp descending, and a `get` returns the newest visible value per qualifier by default.

**Worked example.** Two puts to `i:amount` at timestamps `t1` and `t2` produce two cells in the store. A default `get` returns the `t2` value; `get.addColumn(...).readVersions(3)` can return both plus an older one. A delete adds a third cell, the tombstone.

**Common mistake.** Confusing the cell with a row. A row is a collection of cells across families, and "cell-level atomicity" means one cell's update is atomic, while a multi-cell `Put` on one row is also applied atomically. There is no atomicity across rows.

## What is a Bloom filter?

A Bloom filter is a space-efficient structure used to test whether an HFile contains a certain row or row-column cell, which avoids reading HFiles that cannot contain the requested key. HBase supports `ROW` (test row key only) and `ROWCOL` (test row key plus column) Bloom filters per column family, which improves read throughput for random point reads.

```bash
# Add a Bloom filter to a family, or tune it at create time
alter 'app:impressions', {NAME => 'i', BLOOMFILTER => 'ROWCOL'}
```

Bloom filters help point reads and hurt nothing on writes except a little memory and file size; they do not help scans of many rows.

**Mechanics.** A Bloom filter is a compact bit array per HFile that answers "definitely not present" or "possibly present" for a key, so a reader can skip HFiles that cannot hold the row. HBase builds it at flush and compaction time, stores it in the HFile's metadata blocks, and loads it into the block cache on first use. The two modes are `ROW` (test the row key) and `ROWCOL` (test the row plus column, at a larger size).

**Worked example.** A store with 20 HFiles and a `ROWCOL` Bloom filter answers a point `get` by testing 20 tiny structures instead of reading 20 block indices. HFiles that test negative are skipped entirely, so the read touches only the file that holds the row. The cost is roughly 1 to 2 percent more file size and some cache footprint.

**Common mistake.** Enabling `ROWCOL` on a table read only by whole-row `get`s. `ROW` is smaller and sufficient when you fetch the row, and `ROWCOL` only pays off when reads touch specific columns. Bloom filters also do nothing for range scans, so do not expect them to speed up `scan`.

## Which technique can you use in HBase to access HFile directly without the help of HBase?

You can use the `HFile.main()` method (the `hbase org.apache.hadoop.hbase.io.hfile.HFile` tool) to inspect an HFile without a running HBase cluster.

```bash
hbase org.apache.hadoop.hbase.io.hfile.HFile -p -f /hbase/data/default/app_impressions/region/hfile
```

**Mechanics.** HFiles are self-describing: a file holds data blocks, an index block, a Bloom filter block, and file info. The `HFile` tool reads that metadata without a cluster, using `HFilePrettyPrinter` to print the trailer version, compression codec, key count, and, with `-p`, a sample of key-value pairs.

```bash
hbase org.apache.hadoop.hbase.io.hfile.HFile -f hdfs://nn/hbase/data/.../abc123 -m -p
hbase org.apache.hadoop.hbase.io.hfile.HFile -f <file> -b -e   # block index / stats
```

**Worked example.** Debugging a suspected bad codec: run the tool with `-m` to print the HFile metadata, confirm the compression algorithm, and compare the key count with the expected row count for that region's range.

**Common mistake.** Pointing the tool at a region directory instead of a specific HFile, or assuming it can repair anything. It is an inspection tool; a corrupt HFile is handled by `hbck` or by dropping and rebuilding the region, not by the printer.

## Can the region server will be located on all DataNodes?

Yes. In a normal deployment RegionServers run on the same hosts as DataNodes so that HDFS data locality lets a RegionServer read its regions from local disks.

**Mechanics.** HBase colocates RegionServers with DataNodes so a region's HFiles sit on the same host's disks. That gives data locality: a read served by the local RegionServer is a local disk read rather than a network read. The default deployment makes every DataNode also run a RegionServer, and HDFS replication gives each block three copies on three such hosts.

**Worked example.** A region is served by node A. Its store files are HDFS blocks whose primary replica is on A. A `get` reads from A's local disk. If A dies, the master reassigns the region to node B, which reads the same HDFS blocks over the network until it compacts and rewrites them locally, at which point locality returns.

**Common mistake.** Assuming locality is guaranteed. It degrades after failover, after a bulk load, and after a region move, and it is restored by compaction. Monitor the region locality metric and be aware that a fresh bulk load reads remotely until the first compaction.

## Define the difference between Hive and HBase?

Apache Hive is a data warehousing layer on top of Hadoop. It queries data in HDFS with HiveQL (a SQL-like language) that is translated into MapReduce, Tez, or Spark jobs, and it performs batch processing.

Apache HBase is a NoSQL wide-column key/value store on top of HDFS. Unlike Hive, HBase operations run in real time against its own database rather than as batch jobs. HBase splits tables into regions and groups columns into column families.

Hive and HBase are complementary: Hive is good for analytical scans and joins, HBase for real-time key-based access. Data can be read and written between them, for example by storing HBase tables in Hive or writing Hive results into HBase.

**Mechanics.** Hive is a SQL engine over files in HDFS with a metastore catalogue; queries become MapReduce, Tez, or Spark jobs with minutes of latency. HBase is a random-access store with millisecond gets. They can be bridged: Hive can define an external table over an HBase table using `STORAGE HANDLER` and read it through a scan.

**Worked example.** A daily job in Hive aggregates raw events in Parquet and writes a compact summary back into HBase for serving. Interactive queries hit HBase; heavy reports hit Hive on the file copy, so neither workload starves the other.

**Common mistake.** Choosing Hive-on-HBase for a query that scans the whole table. Hive over HBase cannot push a `LIMIT` into the store the way Phoenix can, so a `SELECT ... LIMIT 10` may still scan everything. Also, Hive's HBase integration is read-mostly and awkward for writes; use Phoenix or the native client for that.

## Explain the data model of HBase.

HBase comprises a set of tables. Each table has rows and column families, and the row key acts as the primary key used for every access. A cell is addressed by row key, column family, qualifier, and timestamp, and each cell can hold multiple versions. Column qualifiers are dynamic per row, so a table can be extremely sparse and wide.

```text
table
  row key (sorted) -> column family -> qualifier -> [version, value]
```

**Mechanics.** The model is a four-level sorted map: table, row key, column family plus qualifier, timestamped value. Row keys are sorted byte-wise and split into regions; families are declared at create time and are the storage unit; qualifiers are created implicitly on first write. A cell holds the value bytes plus a type byte, and versions are ordered newest first.

```text
table -> row key (sorted) -> family -> qualifier -> [(timestamp, value)]
```

**Worked example.** One row can hold two families with different version policies: `i:amount` keeps one version for 90 days, `meta:note` keeps three versions with no TTL. Both live in the same row but in separate stores with separate HFiles.

**Common mistake.** Expecting a schema registry or types. HBase stores bytes, so a `put` of the string `"42.10"` and a `put` of the double `42.1` are different byte arrays. Enforce types in the application or let Phoenix declare them, and be consistent about byte order and encoding across writers.

## What is RegionServer?

A table is divided into regions, and a RegionServer serves a set of regions to clients. It handles reads and writes, keeps the MemStore, appends to the WAL, flushes to HFiles, and runs compactions and splits. In a distributed cluster a RegionServer runs on a DataNode.

**Mechanics.** A RegionServer is the data plane process. It hosts a set of regions, each with one store per family; it appends mutations to the WAL, applies them to MemStores, flushes HFiles, runs compactions and splits, and serves scans and gets from the block cache and HFiles. It registers with ZooKeeper and reports load to the master.

**Worked example.** A node with 8 GB of heap and a 0.4 global MemStore fraction can hold roughly 3.2 GB of MemStore. If 200 regions each hold 32 MB of MemStore, that is 6.4 GB, so the server flushes the largest MemStores to stay under the limit. That is why region count and region size are capacity decisions, not just layout decisions.

**Common mistake.** Treating RegionServers as stateless workers with no capacity limit. Too many regions per server means many small MemStores, constant flushing, and a hot compaction queue. The practical guidance is tens to low hundreds of regions per server, with region sizes in the single-digit GB range.

## What are the data manipulation commands of HBase?

+ `put` - writes a cell value at a specified row and column.
+ `get` - fetches a row or a specific cell.
+ `delete` - deletes a cell value.
+ `deleteall` - deletes all cells in a row.
+ `scan` - scans and returns table data.
+ `count` - counts the rows in a table.
+ `truncate` - disables, drops, and recreates a table.
+ `incr` - atomically increments a counter cell.
+ `append` - appends bytes to an existing cell value.

```bash
# Truncate keeps the table definition but removes all data
truncate 'app:impressions'
```

**Mechanics.** The shell DML commands map to `Table` and `Admin` methods. `put` writes one cell, `get` reads a row, `scan` reads a range, `delete`/`deleteall` write tombstones, `incr` atomically increments a counter cell, `append` appends bytes, and `count` runs a scan that counts rows (optionally with an interval).

```bash
put 'app:impressions', 'u42|1', 'i:campaign', 'cmp-99'
get 'app:impressions', 'u42|1'
scan 'app:impressions', {STARTROW => 'u42|', ENDROW => 'u42|ÿ', LIMIT => 10}
incr 'app:counters', 'daily|2026-10-04', 'c:impressions', 1
delete 'app:impressions', 'u42|1', 'i:campaign'
count 'app:impressions', INTERVAL => 100000
```

**Common mistake.** Using `count` on a large production table. It is a full scan and it contends with serving traffic. For an estimate, use `count` with a large interval or read the region-level metrics and row count estimates instead. Also, `delete` needs a column or timestamp for a cell delete; `deleteall` is the row-level command.

## What is the use of ZooKeeper?

ZooKeeper maintains cluster configuration and the communication path between RegionServers and clients. It also provides distributed synchronization and holds cluster state through sessions. Every RegionServer and the HMaster send heartbeats to ZooKeeper, which detects failures and provides server-failure notifications so recovery can start. Clients also use ZooKeeper to find `hbase:meta`.

**Mechanics.** ZooKeeper provides the cluster's shared state: it stores the location of `hbase:meta`, holds master election (one active, the rest standby), and tracks RegionServer liveness through ephemeral znodes and heartbeats. Clients read `hbase:meta`'s location from a well-known znode and cache it. A `RegionServer` znode set drives split and merge coordination too.

**Worked example.** A RegionServer stops sending heartbeats, its ephemeral znode expires, and ZooKeeper notifies the master, which starts a `ServerCrashProcedure`: split the WAL, reassign the regions, and replay the edits. No operator action is needed.

**Common mistake.** Running three ZooKeeper peers on the same physical hosts as the HBase daemons and calling it fault tolerance. ZooKeeper needs an odd quorum of independent nodes with low latency; a shared host means one failure takes out a peer and a RegionServer. Also, never delete ZooKeeper znodes by hand to "fix" a stuck cluster.

## Define catalog tables in HBase?

Catalog tables maintain metadata about the cluster. `hbase:meta` maps each region to its RegionServer and start row key. In modern HBase the old `-ROOT-` table no longer exists; the location of `hbase:meta` is stored in ZooKeeper, and clients cache region locations locally.

**Mechanics.** `hbase:meta` is the only catalogue table in modern HBase. Its row keys are `tableName,startKey,regionId`, and each row holds the region's `info:regioninfo` (the key range and encoded name) and `info:server` (the RegionServer currently hosting it). It is itself split into regions, so lookups of `hbase:meta` also go through region routing, with the meta regions' locations cached in ZooKeeper.

```bash
scan 'hbase:meta', {LIMIT => 5}
get 'hbase:meta', 'app:impressions,,1'
```

**Worked example.** To find the owner of a key, the client reads the `hbase:meta` row whose start key is the greatest start key not greater than the target row key, gets its `info:server`, and sends the request there.

**Common mistake.** Editing `hbase:meta` directly with `put` or `delete`. The master caches state and would not notice, producing a split brain. Use `hbck`/`hbck2` or admin APIs for consistency repair. Also, `-ROOT-` no longer exists in modern HBase; assuming it does is an outdated answer.

## What are the step in writing something into HBase by a Client?

The client does not write directly to HFiles. It first appends the mutation to the WAL, then applies it to the MemStore of the target region. The MemStore is flushed to disk as an HFile when it reaches a threshold or the region is closed, and the WAL entries for those edits can then be discarded. Region location comes from the client's cache of `hbase:meta`.

```text
put -> find region (meta cache) -> RegionServer -> WAL append -> MemStore -> (flush) HFile
```

**Mechanics.** A client write is a pipeline: resolve the region from the cached `hbase:meta` map, send the mutation to the owning RegionServer, append to the WAL, apply to the region's MemStore, acknowledge. The client never writes an HFile. Flushing happens later, either when a MemStore crosses `hbase.hregion.memstore.flush.size`, when the server's global MemStore limit is reached, or when the region is closed or moved.

```text
put -> meta cache lookup -> RegionServer -> WAL append -> MemStore -> ack
                                            (later) flush -> HFile on HDFS
```

**Worked example.** With a 128 MB flush threshold, a region takes roughly 128 MB of writes between flushes. At 5,000 writes per second of 200 bytes each, that is about 1 MB/second per region, so a flush every two minutes and a new HFile per flush.

**Common mistake.** Assuming the write is durable once it is in the MemStore. It is durable once the WAL append is synced; MemStore-only data is lost on crash unless the WAL replays it. The other common error is expecting a put to be fast without bound: when the MemStore is over the global limit, the RegionServer blocks writes until a flush completes.

## What are the different compaction types in HBase?

There are two types. In minor compaction, a small set of adjacent HFiles is merged into one without removing the deleted data, and the files to merge are chosen by policy. In major compaction, all HFiles of a region's column family are merged into one, deleted and expired cells are discarded, and it is often triggered manually or by a schedule.

```bash
# Trigger a major compaction on a region (shell) or a single table
major_compact 'app:impressions'
```

Major compaction rewrites a lot of data, so it should be scheduled off-peak and rate-limited with `hbase.hstore.compaction.throughput.lower.bound` and related settings.

**Mechanics.** Minor compaction is incremental: it picks adjacent files by the compaction policy, merges them, and leaves tombstones because it cannot prove no other file holds data the tombstone masks. Major compaction is total: all store files of a region's family merge into one, and deletes, TTL-expired cells, and surplus versions are dropped. Major compaction is also the point at which a change like block size, compression, or family TTL is fully materialized.

```bash
major_compact 'app:impressions'              # whole table
major_compact 'app:impressions', 'encoded_region_name'
compact 'app:impressions'                    # enqueue a minor compaction
```

**Worked example.** A region with 30 store files and heavy deletes is read-slow. A minor compaction leaves tombstones and does not reduce the file count much. A scheduled major compaction merges to one file, drops the tombstones, and the read starts hitting a single index and Bloom filter.

**Common mistake.** Major-compacting everything at once on a busy cluster. Major compaction rewrites the whole region, so a cluster-wide sweep saturates disks and can cause write stalls. Use the major compaction interval with jitter, or trigger per-region during low traffic, and rate-limit with the throughput bounds.

## What is the Scope of a Rowkey in HBase?

A row key identifies a row within a table, not within a column family. All column families in that row share the same row key; a row can exist with cells in only some (or none explicitly, if a family marker is written) of the families. Two different column families do not have separate row key spaces.

**Mechanics.** A row key is unique within a table, not within a family. Families are separate stores but they share the row's key, so a "row" is the logical union of its cells across families. There is no per-family key space and no foreign key between families.

**Worked example.** Row `u42` may have a cell in family `p` and none in family `s`; the row still exists, and `get 't','u42'` returns the `p` cell only. A `deleteall` on the row writes markers for both families, so it removes cells that do not currently exist too, which does not matter but does explain why the operation looks family-scoped.

**Common mistake.** Thinking each family has its own rows or that a missing family means a missing row. Absence of cells in a family is normal in a sparse table, and a `get` with no family restriction simply returns what exists.

## What is a Namespace in HBase?

A namespace is a logical grouping of tables, similar to a database or schema in a relational database. It organizes tables and controls quotas and permissions. The default namespaces are `default` and `hbase`.

```bash
create_namespace 'app'
create 'app:impressions', 'i'
```

**Mechanics.** A namespace groups tables and carries quotas and permissions. Table names take the form `namespace:table`, and the ACL and quota commands operate at the namespace level, which reduces per-table administration. The built-in namespaces are `default` (unqualified table names) and `hbase` (system tables).

```bash
create_namespace 'app', {'hbase.namespace.quota.maxregions' => '200'}
create 'app:impressions', {NAME => 'i'}
list_namespace_tables 'app'
describe_namespace 'app'
alter_namespace 'app', {METHOD => 'set', 'hbase.namespace.quota.maxregions' => '400'}
```

**Worked example.** Give each tenant a namespace with a region quota and an ACL granting only that tenant's group access. A runaway tenant then hits `NamespaceRejected` on table creation instead of consuming the whole cluster.

**Common mistake.** Assuming namespaces are separate databases with separate configuration. They share the cluster, the RegionServers, and the memory budget; a quota limits region count or throughput but does not isolate performance. Related mistake: forgetting that `hbase` namespace tables are system tables and should not be changed by hand.

## What is Hotspotting in HBase?

Hotspotting is when a large amount of client traffic is directed at one node, or a few nodes, of a cluster. The traffic can be reads, writes, or other operations, and it overwhelms the machine hosting that region, causing performance degradation and potentially region unavailability. The classic cause is a monotonically increasing row key such as a timestamp or sequential id, which always writes to the last region.

Fixes, in the order an interviewer usually wants to hear them:

+ Salt the row key with a hash prefix so writes spread across regions.
+ Use a reverse timestamp so the newest row is first for a known prefix but writes are not sequential across users.
+ Pre-split the table so the key space starts balanced.
+ Use a random or hashed key when point lookups are by a secondary key.

In [ ]:
import hashlib

def salted_key(user_id: str, ts_ms: int, buckets: int = 32) -> str:
    salt = int(hashlib.md5(user_id.encode()).hexdigest(), 16) % buckets
    return f"{salt:02d}|{user_id}|{9_999_999_999_999 - ts_ms:013d}"

Salting costs read fan-out: a query for one user must know the salt (derive it from the user id) rather than scanning all buckets.

**Mechanics.** A hotspot is a region whose request rate or byte rate far exceeds the cluster average, so its host becomes the bottleneck while other hosts idle. The usual cause is a monotonically increasing key that always appends to the last region; a secondary cause is one very popular key, which no amount of pre-splitting fixes because a single row lives on a single region.

**Worked example.** Diagnose from `hbase:meta` plus the RegionServer metrics: list regions with `list_regions`, then compare per-region `readRequestCount` and `writeRequestCount`. A region with 40 percent of the cluster's requests while the cluster has 60 regions is the hotspot. If the keys within that region are sequential, the write is a key-design problem; if one row dominates, it is a single-key problem.

**Common mistake.** Fixing a hotspot by splitting the region. Splitting a single hot row does nothing, because the row cannot move. Pre-splitting helps before the load arrives, not after a popular key is established; the real fixes are salting and a fan-out read or a rewrite of the key to include a discriminator.

## What is Ttl in HBase?

TTL (time to live) is a data retention setting at the column family level. It specifies how long a cell version is kept; once a version's timestamp is older than the TTL, it is removed during compaction and is not returned by reads.

```bash
create 'app:impressions', {NAME => 'i', TTL => 7776000}   # 90 days in seconds
```

**Mechanics.** TTL is a family-level setting measured in seconds from a cell's timestamp. During reads, versions older than `now - TTL` are not returned. During compaction, they become eligible for removal. TTL therefore reclaims space lazily, at compaction time, not at the moment each cell expires.

```bash
create 'iot:readings', {NAME => 'd', VERSIONS => 1, TTL => 7776000}   # 90 days
alter 'iot:readings', {NAME => 'd', TTL => 2592000}                   # shorten to 30 days
```

**Worked example.** A 90-day TTL table stops returning data older than 90 days immediately after the alter, but the disk usage only falls as regions are compacted. Because the expired cells are dropped at major compaction, storage settles about one compaction cycle after the change.

**Common mistake.** Writing cells with the default server timestamp when the source has an older event time. If the source timestamp is older than the TTL, the cell is invisible on arrival. Set the timestamp explicitly to the event time when you intend TTL to mean event age, and confirm the client is not accidentally writing in seconds versus milliseconds.

## Why do we Pre create empty Regions?

Tables are initially created with one region by default. During bulk import, all clients write to that single region until it grows large enough to split, which serializes the load and slows the import. Pre-creating empty regions (pre-splitting) spreads the initial write load across the cluster from the start.

```bash
create 'app:impressions', {NAME => 'i'}, SPLITS => ['10', '20', '30', '40']
```

**Mechanics.** A new table starts with one region covering the whole key space, so every writer targets one RegionServer until the region splits. Splits are automatic but reactive: they happen after the region exceeds `hbase.hregion.max.filesize`, and each split is followed by compaction of the daughters. Pre-splitting creates the regions up front so the load is parallel from the first write.

```bash
create 'app:impressions', {NAME => 'i'}, SPLITS => ['10','20','30','40','50','60','70','80']
create 'app:impressions', {NAME => 'i'}, SPLITS_FILE => 'splits.txt'
```

**Worked example.** With eight pre-split boundaries there are nine regions, each owning a salt prefix range. If the salted key's first two hex characters are the salt, the pre-split boundaries match the salt distribution, and a bulk load writes to nine regions in parallel instead of one.

**Common mistake.** Pre-splitting with boundaries that do not match the key's leading bytes, which creates empty regions and one overloaded one. Also, pre-splitting into hundreds of tiny regions wastes MemStore budget; size the count so each region starts near one to two gigabytes.

## What are the Two Types of Table Design Approach in HBase?

They are short-and-wide and tall-and-thin.

+ Tall-and-thin: many rows, few columns per row. This is the default HBase sweet spot and scales best because rows spread across regions.
+ Short-and-wide: fewer rows, many columns per row. Useful when a natural entity is read as a whole and the number of entities is bounded, but a single row cannot be split across regions, so it limits write parallelism.

Note the common confusion: "short and wide" means few rows and many columns, not few columns and many rows.

**Mechanics.** Two layout shapes. Tall-and-thin means many rows with few qualifiers each, one row per event, so the key space spreads across regions and writes parallelize. Short-and-wide means few rows with many qualifiers, so one entity is read in a single get, but the row cannot be split across regions and writes to it serialize.

```text
tall-and-thin:  1 row per event  -> rows spread, write-parallel, scans needed
short-and-wide: 1 row per entity -> one get, but a single row lives on one region
```

**Worked example.** Time series is tall-and-thin: `device|reverse_ts` gives one row per reading, and the newest N readings are a bounded prefix scan. A product catalog with per-category attributes can be short-and-wide when products are read whole and updates are rare per product.

**Common mistake.** Choosing short-and-wide for time series or event streams. It concentrates a hot entity in a single region, so ingestion for that entity cannot scale, and a row with millions of qualifiers becomes slow to read and impossible to split.

## When do we do Manual Region Splitting?

Manual region splitting is done when there is an unexpected hotspot caused by many clients querying the same region, or when you know the key distribution ahead of time and want to pre-split for bulk loading. Automatic splitting handles normal growth.

```bash
# Split a table at an explicit key, and merge two adjacent regions
split 'app:impressions', '20'
merge_region 'ENCODED_REGION_A', 'ENCODED_REGION_B'
```

**Mechanics.** Automatic splitting creates a new region when a region's largest store exceeds `hbase.hregion.max.filesize`. Manual splitting is for the cases automation handles badly: a known key distribution before a bulk load, a hot region that should be cut sooner than the size policy would, or regions that need to be merged back after a load spike.

```bash
split 'app:impressions'                       # split the largest region
split 'app:impressions', '30'                 # split at an explicit key
split 'ENCODED_REGION_NAME'
merge_region 'ENCODED_A', 'ENCODED_B'
```

**Worked example.** A bulk load is about to write keys starting `00` through `ff`. Pre-split at `10,20,...,f0` so the load writes to sixteen regions at once. Later, if two adjacent regions are tiny, merge them to reduce region count and MemStore overhead.

**Common mistake.** Splitting a region whose hot key is a single row, which does not help, and splitting repeatedly to chase a hotspot, which produces a split storm. Also, splitting creates daughters that each need compaction, so splitting a cluster-wide table during peak write traffic adds load rather than removing it.

## In HBase what is Log Splitting?

When a RegionServer dies, the edits belonging to its regions must be recovered from the WAL. The edits in the WAL are grouped by region so each set can be replayed to rebuild that region's data. The process of grouping and distributing WAL edits by region is called log splitting.

**Mechanics.** Log splitting happens after a RegionServer failure. The dead server's WAL files are still in HDFS, but each file interleaves edits for all the regions that server hosted. The master (or the `WALProcedureStore` / `SplitLogManager`) reads the WALs, groups entries by region, writes per-region recovered edit files, and assigns each group to the server that now owns that region for replay.

**Worked example.** A server hosting 180 regions dies with 12 WAL files. Splitting produces up to 180 edit files, and each new owner replays only its own. The whole operation runs as a procedure, so a master failover mid-recovery resumes rather than restarting.

**Common mistake.** Confusing log splitting with HDFS block recovery. HDFS handles the durability of the WAL files themselves; splitting is an HBase-level fan-out by region. Also, a single WAL with entries for many regions is exactly why replay is not instant, and why MultiWAL and region grouping exist.

## How does HBase support Bulk Data Loading?

Bulk loading writes HFiles directly, bypassing the WAL and MemStore, which makes it much faster than `put` for large imports. Two steps:

1. Generate HFiles from the source data with a MapReduce or Spark job (`ImportTsv` or a custom job) in HBase's internal format.
2. Load the prepared files into a running cluster with `completebulkload` / `LoadIncrementalHFiles`, which moves each file into the correct region.

```bash
# 1) Convert a TSV on HDFS into HFiles
hbase org.apache.hadoop.hbase.MapReduce.ImportTsv \
  -Dimporttsv.columns=HBASE_ROW_KEY,i:campaign_id,i:amount \
  -Dimporttsv.bulk.output=/tmp/impressions_hfiles \
  app:impressions /data/incoming/impressions.tsv

# 2) Load the HFiles into the table (pre-split it first for balance)
hbase org.apache.hadoop.hbase.MapReduce.LoadIncrementalHFiles \
  /tmp/impressions_hfiles app:impressions
```

Pre-split the table before a bulk load, because `LoadIncrementalHFiles` assigns each HFile to the region that owns its key range.

**Mechanics.** A normal put goes through the WAL and MemStore and later becomes a small HFile on a RegionServer. A bulk load inverts that: a MapReduce or Spark job reads the source and writes HFiles directly in HBase's format, sorted by key, and then `LoadIncrementalHFiles` moves each file into the region that owns its key range. No WAL, no MemStore, no replay, and no compaction for the loaded bytes.

```text
source -> (MapReduce: KeyValueSortReducer) -> HFiles on HDFS
       -> completebulkload / LoadIncrementalHFiles -> region directories
```

**Worked example.** Importing a 2 TB export with plain puts saturates the WAL and triggers constant flushes. The bulk path writes HFiles in parallel, and the load phase is a set of renames. Throughput rises by an order of magnitude and the serving cluster sees almost no write pressure.

**Common mistake.** Bulk-loading into a table with one region. If the table is not pre-split, every HFile lands in the single region and the load phase serializes. Pre-split to match the key distribution first, and remember that the loaded files are not locally replicated until HDFS places them, so locality is briefly worse.

## Why MultiWAL is Needed?

With a single WAL per RegionServer, the RegionServer must write to the WAL serially because HDFS files must be sequential. That makes the WAL a throughput bottleneck. MultiWAL gives each RegionServer multiple WAL files so different regions can write concurrently, improving write throughput.

```xml
<property>
  <name>hbase.wal.regiongrouping.strategy</name>
  <value>identity</value>
</property>
<property>
  <name>hbase.wal.provider</name>
  <value>filesystem</value>
</property>
```

**Mechanics.** An HDFS file is sequential, so a single WAL serializes appends from all regions on a server. MultiWAL gives the RegionServer several WAL files and routes a region's appends to one of them, so independent regions can write concurrently and the single-file bottleneck disappears. The WAL provider and region grouping strategy control how many files exist and how regions map to them.

```xml
<property><name>hbase.wal.provider</name><value>filesystem</value></property>
<property><name>hbase.wal.regiongrouping.strategy</name><value>identity</value></property>
<property><name>hbase.wal.regiongrouping.numgroups</name><value>4</value></property>
```

**Worked example.** A write-heavy server with 300 regions and one WAL caps at the throughput of one HDFS append stream. With four WAL groups, four regions can append concurrently, roughly quadrupling the achievable write rate until another resource becomes the limit.

**Common mistake.** Enabling MultiWAL and expecting it to fix a hotspot or a MemStore-limited workload. It relieves WAL serialization only; if writes block because of flushes or a single hot region, WAL grouping will not help. More WAL groups also mean more files to split on recovery, which lengthens failover.

## How does HBase provide High Availability?

+ Region replication: each region can have replicas served by different RegionServers, and the load balancer avoids co-hosting a replica set.
+ WAL replay: committed edits are recovered from the WAL after a RegionServer crash.
+ Multiple masters: backup H Masters are elected so the master is not a hard single point of failure.
+ HDFS replication: the underlying WALs and HFiles are stored with HDFS replication.
+ For cross-cluster DR, HBase replication ships edits from a source cluster to a peer cluster.

**Mechanics.** Availability is layered. HDFS replication protects the bytes; the WAL plus replay protects acknowledged but unflushed writes; region replicas serve stale reads from a second server when the primary is unavailable; standby masters remove the master as a hard failure point; and cross-cluster replication ships edits for disaster recovery.

```xml
<property><name>hbase.regionserver.storefile.refresh.period</name><value>0</value></property>
<property><name>hbase.region.replica.replication.enabled</name><value>true</value></property>
```

**Worked example.** With region replication of 2, a serving application can set `Consistency.TIMELINE` on a `get` to allow the secondary to answer, trading strict freshness for availability during a failover window. The primary handles writes and the secondary replays edits asynchronously.

**Common mistake.** Confusing HBase region replication with HDFS replication and assuming both give read scaling. Region replicas can serve reads, but they are eventually consistent and primarily an availability feature. Cross-cluster replication is asynchronous and is not a synchronous DR guarantee.

## How does Wal help when a Regionserver Crashes?

The WAL records all changes to data in file-based storage. If a RegionServer crashes or becomes unavailable before the MemStore is flushed, the WAL ensures the changes can be replayed. The master splits the WAL by region and a new RegionServer replays the edits, so acknowledged writes survive the crash.

**Mechanics.** The WAL is the recovery contract: a write is acknowledged only after the WAL append. If the RegionServer dies before the MemStore is flushed, the master learns of the failure from ZooKeeper, splits the dead server's WALs by region, and assigns the regions to new servers, which replay their edits before serving. Sequence numbers let replay skip anything already flushed.

**Worked example.** A server with 4 GB of unflushed MemStore crashes at 12:00:00. Recovery splits the WAL, replays each region's edits on the new owner, and by 12:00:30 the regions are serving with the acknowledged writes present. Clients see a short retry window, not data loss.

**Common mistake.** Assuming recovery is instantaneous or invisible. The WAL for a busy server can be gigabytes, and log splitting plus replay takes time proportional to that size, which is why region count per server, flush tuning, and MultiWAL grouping matter for failover time. Writes made with `ASYNC_WAL` or `SKIP_WAL` are genuinely lost.

## What are the different Block Caches in HBase?

HBase provides two BlockCache implementations: the on-heap `LruBlockCache` (the default) and `BucketCache`, which is usually off-heap. A common production configuration is a combined cache: an LRU cache in front of a `BucketCache` backed by off-heap memory, SSD, or a file, to keep the JVM heap small and avoid GC pauses.

```xml
<property>
  <name>hbase.bucketcache.ioengine</name>
  <value>offheap</value>
</property>
<property>
  <name>hbase.bucketcache.size</name>
  <value>16384</value>
</property>
<property>
  <name>hfile.block.cache.size</name>
  <value>0.2</value>
</property>
```

**Mechanics.** There are two on-heap/off-heap designs. `LruBlockCache` is a single on-heap LRU segmented by priority (single, multi, in-memory access). `BucketCache` stores blocks in fixed-size buckets off-heap, on SSD, or in a file, which keeps them out of the JVM heap and avoids GC pressure. The common production setup is a small LRU cache in front of a larger BucketCache.

```xml
<property><name>hfile.block.cache.size</name><value>0.2</value></property>
<property><name>hbase.bucketcache.ioengine</name><value>offheap</value></property>
<property><name>hbase.bucketcache.size</name><value>16384</value></property>
```

**Worked example.** A 32 GB heap with a 0.2 LRU fraction gives about 6.4 GB of cache; adding a 16 GB off-heap BucketCache raises the effective cache to about 22 GB without enlarging the heap, so a working set that previously missed now hits.

**Common mistake.** Setting `hfile.block.cache.size` too high. The MemStore fraction and the cache fraction must leave GC headroom, and on a cache-heavy read workload the block cache is the one to grow first. Also note that the block cache holds HFile blocks, not rows, and that a cold cache after restart makes latency spike until it warms.

## How do you manage compactions and region splits?

Compaction and splitting are the two background processes that keep a growing table healthy. Compaction controls read amplification and storage; splitting controls distribution across RegionServers. Both are usually automatic, but the commands and policies matter in interviews.

```bash
# Inspect and trigger compactions
major_compact 'app:impressions'
compact 'app:impressions'
# Split and merge regions
split 'app:impressions', '30'
merge_region 'ENCODED_REGION_A', 'ENCODED_REGION_B'
```

```xml
<property>
  <name>hbase.hregion.majorcompaction</name>
  <value>604800000</value>
</property>
<property>
  <name>hbase.hregion.majorcompaction.jitter</name>
  <value>0.5</value>
</property>
<property>
  <name>hbase.hstore.compaction.min</name>
  <value>3</value>
</property>
<property>
  <name>hbase.hstore.blockingStoreFiles</name>
  <value>16</value>
</property>
<property>
  <name>hbase.regionserver.region.split.policy</name>
  <value>org.apache.hadoop.hbase.regionserver.SteppingSplitPolicy</value>
</property>
```

Practical guidance:

+ Schedule major compactions off-peak and jitter them, because they rewrite the whole region.
+ On a write-heavy table, tune `hbase.hstore.compaction.min` and the throughput bounds rather than disabling compaction.
+ Pre-split new tables and use `SteppingSplitPolicy` to avoid split storms, where one split immediately triggers the next.
+ A region that stays hot after its split is a row key design problem, not a compaction problem.

**Mechanics.** Compaction is governed by the store file count and the compaction policy; splitting by `hbase.hregion.max.filesize` and the split policy. Both run in the background. `hbase.hstore.blockingStoreFiles` is the back-pressure valve: above it, the server blocks writes for that region so reads do not degrade without bound.

```xml
<property><name>hbase.hregion.max.filesize</name><value>10737418240</value></property>
<property><name>hbase.hregion.majorcompaction</name><value>604800000</value></property>
<property><name>hbase.hregion.majorcompaction.jitter</name><value>0.5</value></property>
<property><name>hbase.hstore.compaction.min</name><value>3</value></property>
<property><name>hbase.regionserver.region.split.policy</name><value>org.apache.hadoop.hbase.regionserver.SteppingSplitPolicy</value></property>
```

**Worked example.** A table with uniform write growth and a 10 GB max file size splits at roughly 10 GB per region. `SteppingSplitPolicy` splits the daughter at half the parent's size, which spreads a growing table faster than `IncreasingToUpperBoundRegionSplitPolicy` and avoids one giant region dominating a server.

**Common mistake.** Setting `hbase.hregion.majorcompaction` to 0 to disable automatic major compaction and then never scheduling one. Tombstones and TTL-expired cells accumulate forever, so storage and read amplification grow monotonically. If you disable the timer, run a scheduled `major_compact` yourself during off-peak hours.

## When should you not use HBase, and what should you use instead?

+ Analytical SQL over the whole dataset - use a warehouse (Snowflake, BigQuery, Redshift) or a lakehouse table (Iceberg, Delta, Hudi) on Parquet. HBase scans are not columnar and do not support joins.
+ Low-latency key-value at very high write rates with multi-region writes - consider Cassandra or Bigtable, which offer multi-datacenter replication without the RegionServer/ZooKeeper stack.
+ Ordered streaming with replay and many consumers - use Kafka as the backbone, not HBase.
+ Small or medium datasets that fit in a relational database - use PostgreSQL or MySQL; HBase only pays off at scale.
+ Mostly write-once time series analytics - use Parquet/Iceberg with time partitioning; HBase is for point access, not aggregation.
+ Strong multi-row ACID transactions - use a relational or NewSQL database, not HBase.

A useful rule: HBase serves a known key to an application. If the question is "what does this key hold right now", HBase is a candidate; if the question is "aggregate everything matching a predicate", it is not.

**Mechanics.** The disqualifiers are the mirror image of the fit criteria. No native joins or multi-row transactions, no secondary index without Phoenix or a hand-built index table, no columnar compression for analytics, and no cheap full-table aggregation. Operational cost is real: ZooKeeper, HDFS, compaction, and region tuning all need owners.

```text
analytics over everything     -> warehouse / lakehouse (Parquet, Iceberg)
multi-region key-value writes -> Cassandra or Bigtable
ordered replay, many readers  -> Kafka, not HBase
fits in one relational node   -> PostgreSQL or MySQL
multi-row ACID                -> relational or NewSQL
```

**Worked example.** A team wants "real-time analytics" and proposes HBase. After listing the queries, most are `GROUP BY` over a time range, so the design becomes Parquet on object storage with a warehouse on top, plus HBase only for the point-lookup serving path if latency demands it.

**Common mistake.** Choosing HBase because it is in the Hadoop stack or because the data is large. Size is not the criterion; the access pattern is. The reverse mistake is also an error: forcing a high-write key-value workload into a relational database and then hand-sharding it, which is the problem HBase was built to solve.

## How do you design a row key for even distribution and diagnose a hotspot?

**Mental model.** HBase has exactly one index: the row key. A region owns a contiguous key range, and the RegionServer hosting that region owns all traffic for that range. Even distribution is therefore a property of the key bytes, not of the table or the cluster. Anything that increases monotonically concentrates load in the last region; anything with a head byte that is spread over the key space distributes load.

**How it works.** Three standard transforms. Salting prepends a fixed-width bucket derived from the entity id, so writes for an entity always land in the same known bucket and different entities spread across buckets. Hashing replaces a natural key with a digest, which spreads writes but destroys range scans. Reverse timestamps make the newest row sort first inside a known prefix, which turns "latest N" into a bounded scan. Pick the transform from the query pattern.

```python
import hashlib

def salted_key(entity_id: str, ts_ms: int, buckets: int = 32) -> str:
    salt = int(hashlib.md5(entity_id.encode()).hexdigest(), 16) % buckets
    return f"{salt:02d}|{entity_id}|{9_999_999_999_999 - ts_ms:013d}"

def hotspot_ratio(counts: dict) -> tuple:
    total = sum(counts.values())
    average = total / len(counts)
    busiest = max(counts, key=counts.get)
    return busiest, counts[busiest] / average
```

**Worked example.** A keyset of sequential device ids produces a perfectly even write distribution once salted, but a read for one device must compute the entity's bucket (`md5(id) % 32`) rather than scanning all 32. If the caller cannot compute the salt, store an index row that maps the entity to its bucket, or accept a 32-way fan-out scan.

**Common mistake.** Salting with too few buckets, or with a bucket derived from something that does not exist at read time. Two or four buckets just move the hotspot; a salt the reader cannot reproduce turns every lookup into a full-table scan.

**Diagnosing a hotspot.** Compare per-region request counts. A region above roughly three times the cluster average, or one server carrying far more regions than its peers, is the suspect. Look at `writeRequestCount` and `readRequestCount` per region, not just cluster totals.

```bash
# Find the region ranges and their server, then compare request rates
list_regions 'app:impressions'
status 'detailed'

# Ask one RegionServer for its region-level metrics over JMX
curl -s "http://rs1.example.com:16030/jmx?qry=Hadoop:service=HBase,name=RegionServer,sub=Regions" | head

# If a single sequential region is hot, split it; if one key is hot, splitting will not help
split 'app:impressions', '40'
```


## How do you size regions, pre-split a table, and choose a split policy?

**Mental model.** A region is the unit of distribution, so region count per server sets the parallelism and the number of MemStores, and region size sets how long a split and a compaction take. Too few regions means one server cannot use its disks and CPUs; too many small regions means many small MemStores, constant flushing, and a heavy compaction queue.

**How it works.** `hbase.hregion.max.filesize` is the store size at which a region splits, so target region size is roughly that value. Region count follows from the data size: `regions = ceil(stored bytes / target region size)`. Pre-splitting passes explicit boundaries at create time, and the boundaries must match the leading bytes of the real row keys. The split policy decides how a daughter splits later; `SteppingSplitPolicy` halves the threshold at each generation, which spreads a fast-growing table quickly, while `IncreasingToUpperBoundRegionSplitPolicy` grows the threshold with the region count.

```xml
<property><name>hbase.hregion.max.filesize</name><value>10737418240</value></property>
<property><name>hbase.regionserver.region.split.policy</name>
  <value>org.apache.hadoop.hbase.regionserver.SteppingSplitPolicy</value></property>
<property><name>hbase.regionserver.regionSplitLimit</name><value>1000</value></property>
```

**Worked example.** 2 billion rows at 400 bytes before compression and a 0.4 compression ratio is 320 GB stored. At a 10 GiB target, that is 30 regions, so pre-split with 29 boundaries spread evenly over the salt range, for example `['10','20',...,'e0']` for a two-character hex salt.

**Common mistake.** Pre-splitting with even byte boundaries when the key is not uniformly distributed across those bytes. If 90 percent of keys start `0`, even boundaries produce one huge region and several empty ones. Derive boundaries from the actual key histogram or from the salt range, and remember that splits and pre-splits both trigger compaction of the new daughters.

```bash
# Pre-split on the salt range, then verify the region chain
create 'iot:readings', {NAME => 'd', VERSIONS => 1, TTL => 7776000}, \
  SPLITS => ['04','08','0c','10','14','18','1c','20','24','28','2c','30']
list_regions 'iot:readings'

# Force a split of one region at an explicit key when a range outgrows its neighbours
split 'iot:readings', '18'
# Merge two small adjacent regions back after a load spike
merge_region 'ENCODED_REGION_A', 'ENCODED_REGION_B'
```


## How do you tune the read path?

**Mental model.** A read merges the region's MemStore with every store file it might need, applying Bloom filters and the block cache to avoid touching files that cannot hold the key. Read latency therefore scales with the number of store files, the block cache hit ratio, and how precisely the request is expressed. Tuning the read path means giving the cache the right working set, giving the reader the right key range, and keeping the file count low.

**How it works.** The block cache holds HFile blocks, so its size relative to the hot working set decides most point-read latency. Bloom filters skip files per key. `Scan.setCaching` sets rows per RPC and `Scan.setBatch` sets cells per RPC, trading memory and round trips. A bounded `STARTROW`/`ENDROW` plus a filter that examines the key is far cheaper than a value filter, because value filters must read the value first.

```xml
<property><name>hfile.block.cache.size</name><value>0.2</value></property>
<property><name>hbase.bucketcache.ioengine</name><value>offheap</value></property>
<property><name>hbase.bucketcache.size</name><value>16384</value></property>
```

```java
Scan scan = new Scan()
    .withStartRow(Bytes.toBytes("07|u42|"))
    .withStopRow(Bytes.toBytes("07|u42|￿".getBytes()))
    .setCaching(200)     // rows per RPC
    .setBatch(100)       // cells per RPC
    .setLimit(1000);     // stop across regions
scan.addFamily(Bytes.toBytes("i"));   // never read families you do not need
```

**Worked example.** A 50-row page over a key range costs one RPC per region when `setCaching(200)`, versus one RPC per row with the default caching of 1. On a 60-region table, the difference is 60 round trips instead of 3,000 for the same page.

**Common mistake.** Raising caching without bound. Large caching means the server holds many rows in memory per scanner, and too many concurrent shallow scanners exhaust the RegionServer. Match caching to row size and limit the number of open scanners. A second mistake is using `BLOCKSIZE` or Bloom filters to fix a scan-latency problem that is really too many store files; compaction and a bounded range fix that.

```sql
-- Phoenix: confirm the query becomes a range scan, not a full scan
EXPLAIN SELECT campaign, amount FROM impressions WHERE user_id = 'u42' AND ts > 1700000000000;

-- A query that does not lead with the primary key needs a secondary index
CREATE INDEX IF NOT EXISTS impressions_campaign_idx ON impressions (campaign) INCLUDE (amount);

-- Cover the query so the index table alone answers it
SELECT campaign, SUM(amount) FROM impressions WHERE campaign = 'cmp-99' GROUP BY campaign;
```


## How do you tune the write path?

**Mental model.** A write is a WAL append plus a MemStore insert, acknowledged before anything reaches an HFile. Throughput is limited by WAL serialization, by MemStore budgets and flush behavior, and by files piling up behind compaction. When the global MemStore limit is crossed, the RegionServer blocks writes, so a compaction backlog eventually becomes a latency problem, not a disk problem.

**How it works.** `hbase.regionserver.global.memstore.size` caps total MemStore use and, when the lower limit is crossed, forces flushes. `hbase.hregion.memstore.flush.size` sets the per-region flush threshold. `hbase.hstore.blockingStoreFiles` blocks writes for a region when too many files accumulate. `hbase.hregion.majorcompaction` schedules major compaction and `hbase.hregion.majorcompaction.jitter` spreads it. WAL compression and MultiWAL grouping reduce the append bottleneck. For a custom loader, `HFileOutputFormat2.configureIncrementalLoad` partitions the reducer output by region boundaries, so each reducer writes HFiles that belong to one region and can be moved in by `LoadIncrementalHFiles`.

```xml
<property><name>hbase.regionserver.global.memstore.size</name><value>0.4</value></property>
<property><name>hbase.regionserver.global.memstore.size.lower.limit</name><value>0.95</value></property>
<property><name>hbase.hregion.memstore.flush.size</name><value>134217728</value></property>
<property><name>hbase.hstore.blockingStoreFiles</name><value>16</value></property>
<property><name>hbase.hregion.majorcompaction</name><value>604800000</value></property>
<property><name>hbase.hregion.majorcompaction.jitter</name><value>0.5</value></property>
<property><name>hbase.wal.regiongrouping.numgroups</name><value>4</value></property>
<property><name>hbase.regionserver.wal.enablecompression</name><value>true</value></property>
```

**Worked example.** A 32 GB heap with a 0.4 MemStore fraction allows about 12.8 GB of MemStore. At a 128 MB flush threshold that is roughly 100 region MemStores of headroom. If the table has 400 regions, average MemStore size must stay near 32 MB or flushes become continuous, which is why region count per server is a memory decision.

**Common mistake.** Raising `hbase.hregion.memstore.flush.size` to reduce flush frequency without checking the global fraction. Larger per-region MemStores mean fewer regions fit in the budget, and the server starts blocking writes sooner under a burst. The other mistake is disabling major compaction to protect throughput and never scheduling it, so tombstones and expired cells accumulate and read amplification grows without bound.

```bash
# Load a large export with HFiles, bypassing the WAL and MemStore
hbase org.apache.hadoop.hbase.mapreduce.ImportTsv \
  -Dimporttsv.separator=, \
  -Dimporttsv.columns=HBASE_ROW_KEY,i:campaign_id,i:amount \
  -Dimporttsv.bulk.output=/tmp/impressions_hfiles \
  app:impressions /data/incoming/impressions.tsv

# Pre-split first, then move the HFiles into the owning regions
hbase org.apache.hadoop.hbase.mapreduce.LoadIncrementalHFiles /tmp/impressions_hfiles app:impressions

# Verify balance and trigger maintenance
list_regions 'app:impressions'
compact 'app:impressions'
major_compact 'app:impressions'
```


## Things to remember: key numbers and limits


| Topic | Fact | Why it matters |
|---|---|---|
| Row key | byte-sorted; regions split on key ranges | a sequential key (timestamp-first) writes to one region — hotspotting |
| Write path | WAL first, then memstore; memstore flush writes HFiles | the WAL is what survives a RegionServer crash |
| Read path | block cache -> memstore -> HFiles with Bloom filters skipping misses | Bloom filters and block cache are why point reads stay fast |
| Deletes | tombstone markers; space reclaimed at major compaction | a delete is a write until compaction removes it |
| Compaction | minor merges HFiles; major rewrites a store fully | major compactions are IO storms; schedule them off-peak |
| Regions | pre-split empty tables; one region serves one key range per server | pre-splitting before a bulk load avoids the single-region collapse |
| TTL/versions | per column family; versions bounded by keeping count or TTL | unbounded versions turn time series into unbounded rows |
| SQL | none natively; Apache Phoenix adds SQL on top | HBase itself is get/scan/put/delete |


## Things to remember: semantics to memorize


* **The row key is the only index.** There are no secondary indexes and no joins; access patterns must be designed into salted, reversed, or composite keys.
* **Tall-narrow vs short-wide is the schema decision.** Tall-narrow (many rows, key varies by event) scales better; short-wide packs versions into one row.
* **A RegionServer dying replays its WAL.** Log splitting assigns the dead server's edits live regions — MultiWAL parallelizes those writes.
* **Bloom filters answer 'not here' fast.** Row-level and row-column Blooms let reads skip HFiles that cannot contain the key.
* **Column families are the physical boundary.** Keep one or two; each family flushes and compacts separately, and many families cause write amplification.


## Things to remember: mistakes that keep coming back


* Timestamp-first row keys — every write hits the newest region; salt or reverse (`Long.MAX_VALUE - ts`).
* Bulk-loading into a single region — the table starts with one region; pre-split on the expected key distribution.
* Many column families — each flush touches only its family but a flush elsewhere flushes all memstores' counts; keep families to one or two.
* Trusting a delete to free space — tombstones persist until major compaction.
* Scans without start/stop keys or filters pushed to the server — the client drags every row over the wire.
* Unbounded versions on a write-heavy family — versions are stored until GC'd by policy or TTL.
* Joining in client code — HBase does not join; denormalize or move the join to Phoenix/Spark and accept what each does.


## Real world use cases for HBase

### 1. IoT device telemetry time series

A utility monitors 60,000 smart meters emitting a reading every 15 seconds. The serving API needs the latest readings for a device and a time-range view over the last day, and the raw history must be retained for 90 days.

Business context: outage detection and customer-facing usage dashboards read a single device at a time. Data volume: about 4,000 writes/second, roughly 200 GB/month after compression.

Row key design: `salt|device_id|reverse_timestamp`, where salt is a fixed hash bucket per device and the reverse timestamp makes the newest reading sort first. TTL on the column family drops data after 90 days.

```bash
# Table with a salted key, one version, 90-day TTL, and a row+column Bloom filter
create 'iot:readings', {NAME => 'd', VERSIONS => 1, TTL => 7776000, BLOOMFILTER => 'ROWCOL', COMPRESSION => 'SNAPPY'}, SPLITS => ['04', '08', '0c', '10', '14', '18', '1c', '20']
```

In [ ]:
import hashlib

def reading_key(device_id: str, ts_ms: int, buckets: int = 32) -> str:
    salt = int(hashlib.md5(device_id.encode()).hexdigest(), 16) % buckets
    return f"{salt:02d}|{device_id}|{9_999_999_999_999 - ts_ms:013d}"

Why this design: the salt spreads writes across all 32 buckets so no single region takes the load, the device prefix within a bucket makes the latest-reading Get a single point read, and the reverse timestamp makes `scan row_prefix=salt|device` return newest first with a `LIMIT`.

### 2. Clickstream profile store for real-time bidding

An ad-tech platform records 2 billion impressions per day and must answer "what did this user see in the last 30 minutes" in under 10 ms during an auction.

Business context: the bidder enriches a request with a user's recent behavior. Data volume: about 25,000 writes/second, 2 billion rows/day, 30-minute working set.

Row key design: `salt|user_id|reverse_timestamp` for the recent window, plus a separate compacted table `user_profile` keyed by `salt|user_id` for the latest aggregates. The recent table uses a short TTL and the serving table keeps one version per key.

```java
Table recent = conn.getTable(TableName.valueOf("rtb:user_events"));
Scan scan = new Scan()
    .withStartRow(Bytes.toBytes(saltOf(userId) + "|" + userId + "|"))
    .withStopRow(Bytes.toBytes(saltOf(userId) + "|" + userId + "|\uFFFF".getBytes()))
    .setLimit(200)
    .setCaching(200);
```

Why this design: point reads and short scans are exactly HBase's strength, the reverse timestamp gives the newest events first without sorting, and the narrow row key keeps the block cache effective. Analytics still runs on a Parquet copy, not on HBase.

### 3. Messaging and conversation history

A messaging platform stores chat history so a client can open a conversation and load the most recent messages, then page backwards.

Business context: 40 million monthly users, peak 50,000 messages/second, per-conversation ordering required. Data volume: several TB per month.

Row key design: `conversation_id|reverse_timestamp|message_id`. Because a conversation is a natural read unit, the design is tall-and-thin with one row per message; the reverse timestamp gives pagination in both directions (newest first, and older pages by advancing the start row).

```bash
# Newest 50 messages in a conversation
scan 'msg:history', {STARTROW => 'conv-88217|', ENDROW => 'conv-88217|\xff', LIMIT => 50}
```

Why this design: ordering by row key replaces an ORDER BY, pagination is a bounded scan, and atomic increments on a companion counters table provide unread counts without a transaction.

### 4. Wide sparse product catalog with per-attribute updates

A retailer stores product attributes where different categories have wildly different fields (electronics have voltages, apparel has sizes), and the pricing service updates individual attributes frequently.

Business context: the storefront reads one product at a time and updates a few attributes per event. Data volume: 50 million products, 10-40 attributes each, low-latency reads.

Row key design: `salt|sku`, with attributes as dynamic qualifiers in a single family. Sparse storage means a product only occupies the qualifiers it actually has.

```bash
put 'catalog:products', '07|sku-99321', 'p:color', 'black'
put 'catalog:products', '07|sku-99321', 'p:voltage', '220'
get 'catalog:products', '07|sku-99321'
```

Why this design: a relational schema would need dozens of sparse columns or an entity-attribute-value table with joins; HBase stores sparse rows natively and reads a whole product in one Get. Salt keeps the catalog write load balanced during bulk price updates.

### 5. Streaming feature store for fraud detection

A payments company computes features (transaction velocity, recent declines) in a stream processor and serves them to a scoring service with single-digit-millisecond latency.

Business context: the scorer runs on every authorization and cannot wait for a warehouse. Data volume: 8,000 events/second, features for 30 million cards.

Row key design: `salt|card_id` for the current feature vector, with the feature values in one family and short TTLs on the fast-moving counters. A separate `salt|card_id|reverse_timestamp` table holds the recent event trail.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Stream processor writes the latest feature vector; scorer reads it
table.put(
    (salt(card_id) + "|" + card_id).encode(),
    {
        b"f:txn_count_1h": str(count_1h).encode(),
        b"f:declines_24h": str(declines_24h).encode(),
        b"f:updated_at": str(now_ms).encode(),
    },
)

Why this design: HBase gives a durable, horizontally scalable, low-latency serving layer for a keyed feature vector, writes are single-row atomic so the vector is never half-updated, and the analytics side of the feature computation lives in Kafka plus Flink rather than in HBase.

**Mechanics.** The case studies below all follow the same design sequence: identify the dominant lookup, choose the row key that makes that lookup a bounded scan, spread writes with a salt or hash, pick one to three families by access pattern and write frequency, set TTL and versions per family, and pre-split to the expected key distribution. Analytics always runs off a copy, never on the serving cluster.

```text
1. dominant lookup  -> 2. row key (order + distribution) -> 3. families (access pattern)
4. TTL / versions   -> 5. pre-split boundaries           -> 6. copy to Parquet for analytics
```

**Worked example.** The IoT case keys on `salt|device_id|reverse_ts` so the latest reading is a point get and the recent window is a bounded prefix scan, uses one family with a 90-day TTL and a `ROWCOL` Bloom filter, and pre-splits on the salt range so the 4,000 writes per second spread evenly.

**Common mistake.** Designing the key from the source data model instead of the query. A natural id that sorts monotonically hotspots; a compound key that leads with a low-cardinality column makes every scan wide. Sketch the two or three real queries first, then write the key that serves them and nothing more.

## HBase exercises

These exercises use only the Python standard library and model HBase sizing and row key decisions with small, runnable calculations. Each one states the task, gives a hint, and names the expected result before the code cell. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are engineering arithmetic, not benchmarks: bytes and regions for sizing, request ratios for hotspot detection, RPC counts for scan cost. Treat the asserted values as a check on the formula.

### Try it 1. Design a salted row key from key patterns

**Task.** Build a salted, reverse-timestamp row key for a set of device ids, verify that the salt is fixed width, that the key is deterministic, and that the ids spread over several buckets.

**Hint.** Derive the salt with `hashlib.md5(entity_id.encode())` modulo the bucket count, format it as `%02d`, and reverse the timestamp with `9_999_999_999_999 - ts_ms`.

**Expected result.** Every key starts with a two-digit salt below 16, the keys are unique, and at least 6 of the 16 buckets are used.

In [ ]:
import hashlib

BUCKETS = 16


def salted_key(entity_id, ts_ms):
    salt = int(hashlib.md5(entity_id.encode()).hexdigest(), 16) % BUCKETS
    return f"{salt:02d}|{entity_id}|{9_999_999_999_999 - ts_ms:013d}"


device_ids = [f"device-{i:04d}" for i in range(1, 17)]
row_keys = [salted_key(device_id, 1_700_000_000_000 + i) for i, device_id in enumerate(device_ids)]

for device_id, row_key in zip(device_ids, row_keys):
    print(f"{device_id} -> {row_key}")

salts = [row_key[:2] for row_key in row_keys]
print("distinct buckets used:", len(set(salts)))

assert all(len(salt) == 2 and 0 <= int(salt) < BUCKETS for salt in salts)
assert len(set(row_keys)) == len(device_ids)
assert salted_key("device-0001", 1_700_000_000_000) == row_keys[0]
assert len(set(salts)) >= 6


### Try it 2. Compute region count from target region size

**Task.** Given a row count, an average row size, a compression ratio, and a target region size, compute the stored bytes, the region count, and the number of pre-split boundaries.

**Hint.** Stored bytes are `rows * avg_row_bytes * compression_ratio`. Use ceiling division `-(-a // b)` so a partial region still counts, and pre-split boundaries are `regions - 1`.

**Expected result.** `320,000,000,000` stored bytes (about `298 GiB`), 30 regions, and 29 pre-split boundaries.

In [ ]:
ROWS = 2_000_000_000
AVG_ROW_BYTES = 400
COMPRESSION_RATIO = 0.4
TARGET_REGION_BYTES = 10 * 1024 ** 3

stored_bytes = int(ROWS * AVG_ROW_BYTES * COMPRESSION_RATIO)
regions = int(-(-stored_bytes // TARGET_REGION_BYTES))

print(f"stored bytes   = {stored_bytes:,}")
print(f"stored GiB     = {stored_bytes / 1024 ** 3:.1f}")
print(f"regions        = {regions}")
print(f"pre-split keys = {regions - 1}")

assert stored_bytes == 320_000_000_000
assert regions == 30
assert regions - 1 == 29


### Try it 3. Detect a hot region from request counts

**Task.** Given per-region request counts, compute the cluster average, the busiest region's ratio to the average, and the list of regions that exceed twice the average.

**Hint.** Ratio is `count / average`. Use `max(counts, key=counts.get)` for the busiest region and a list comprehension for the outliers.

**Expected result.** The busiest region is `region-a` at about `2.97x` the average, and it is the only region above the `2.0x` threshold.

In [ ]:
request_counts = {
    "region-a": 50_000,
    "region-b": 20_000,
    "region-c": 18_000,
    "region-d": 12_000,
    "region-e": 10_000,
    "region-f": 9_000,
    "region-g": 8_000,
    "region-h": 7_500,
}

total = sum(request_counts.values())
average = total / len(request_counts)
ratios = {region: count / average for region, count in request_counts.items()}
busiest = max(ratios, key=ratios.get)
hot_regions = [region for region, ratio in ratios.items() if ratio > 2.0]

print(f"total requests = {total:,}")
print(f"average        = {average:,.1f}")
print(f"busiest        = {busiest} at {ratios[busiest]:.2f}x")
print("regions above 2.0x:", hot_regions)

assert total == 134_500
assert round(ratios[busiest], 2) == 2.97
assert hot_regions == ["region-a"]


### Try it 4. Size MemStore from a heap fraction

**Task.** A RegionServer has a 32 GB heap, a 0.4 global MemStore fraction, a 0.2 block cache fraction, and a 128 MB per-region flush threshold. Compute the MemStore budget, the block cache budget, the GC headroom, and how many region MemStores fit in the budget.

**Hint.** The fractions are relative to the heap, so the budgets are `heap * fraction`. Region MemStore capacity is `memstore_gb * 1024 // flush_size_mb`, and headroom is what is left after both budgets.

**Expected result.** 12.8 GB of MemStore, 6.4 GB of block cache, 12.8 GB of headroom, and 102 region MemStores at the flush threshold.

In [ ]:
HEAP_GB = 32
MEMSTORE_FRACTION = 0.4
BLOCK_CACHE_FRACTION = 0.2
FLUSH_SIZE_MB = 128

memstore_gb = HEAP_GB * MEMSTORE_FRACTION
block_cache_gb = HEAP_GB * BLOCK_CACHE_FRACTION
headroom_gb = round(HEAP_GB - memstore_gb - block_cache_gb, 1)
region_memstores = int(memstore_gb * 1024 // FLUSH_SIZE_MB)

print(f"memstore budget    = {memstore_gb:.1f} GB")
print(f"block cache budget = {block_cache_gb:.1f} GB")
print(f"GC headroom        = {headroom_gb:.1f} GB")
print(f"region MemStores   = {region_memstores} at {FLUSH_SIZE_MB} MB each")
print(f"flush size in bytes = {FLUSH_SIZE_MB * 1024 * 1024}")

assert memstore_gb == 12.8
assert block_cache_gb == 6.4
assert headroom_gb == 12.8
assert region_memstores == 102
assert FLUSH_SIZE_MB * 1024 * 1024 == 134_217_728


### Try it 5. Estimate scan cost from row count and caching

**Task.** Compare the RPCs and bytes of a full-table scan with a bounded prefix scan. The table has 50 million rows of 200 bytes each across 60 regions; the scan uses `setCaching(200)` and the bounded read stops after 1,000 rows.

**Hint.** RPCs are `ceil(rows / caching)` and bytes are `rows * row_bytes`. The bounded case uses the limit instead of the full row count.

**Expected result.** The full scan needs 250,000 RPCs and 10,000,000,000 bytes, the bounded scan needs 5 RPCs and 200,000 bytes, a reduction of 50,000 times.

In [ ]:
import math

ROWS = 50_000_000
ROW_BYTES = 200
CACHING = 200
REGIONS = 60
LIMIT = 1_000

full_bytes = ROWS * ROW_BYTES
full_rpcs = math.ceil(ROWS / CACHING)
bounded_bytes = LIMIT * ROW_BYTES
bounded_rpcs = math.ceil(LIMIT / CACHING)
reduction = full_rpcs // bounded_rpcs

print(f"rows per region      = {ROWS / REGIONS:,.0f}")
print(f"full scan            = {full_rpcs:,} RPCs, {full_bytes / 1024 ** 3:.1f} GiB")
print(f"bounded prefix scan  = {bounded_rpcs} RPCs, {bounded_bytes / 1024:.0f} KiB")
print(f"RPC reduction        = {reduction:,}x")

assert full_rpcs == 250_000
assert full_bytes == 10_000_000_000
assert bounded_rpcs == 5
assert bounded_bytes == 200_000
assert reduction == 50_000


### Try it 6. Choose a Bloom filter type from read patterns

**Task.** Write a function that returns the right Bloom filter setting for an access pattern: whole-row point gets, point gets of a few columns in a wide row, or range and prefix scans.

**Hint.** `ROW` filters on the row key, `ROWCOL` filters on row plus column, and Bloom filters do not help a scan of many rows, so `NONE` is correct there.

**Expected result.** The four patterns map to `ROW`, `ROWCOL`, `NONE`, and `NONE`.

In [ ]:
def choose_bloom(access_pattern):
    if access_pattern == "point get of the whole row":
        return "ROW"
    if access_pattern == "point get of a few columns in a wide row":
        return "ROWCOL"
    if access_pattern in ("range scan", "prefix scan"):
        return "NONE"
    raise ValueError(f"unknown access pattern: {access_pattern}")


patterns = [
    "point get of the whole row",
    "point get of a few columns in a wide row",
    "range scan",
    "prefix scan",
]
choices = {pattern: choose_bloom(pattern) for pattern in patterns}
for pattern, choice in choices.items():
    print(f"{pattern:<45} -> {choice}")

assert choices["point get of the whole row"] == "ROW"
assert choices["point get of a few columns in a wide row"] == "ROWCOL"
assert choices["range scan"] == "NONE"
assert choices["prefix scan"] == "NONE"


### Try it 7. Plan a bulk load

**Task.** A 2,000 GiB export compresses to a 0.4 ratio in HFiles. Plan a bulk load with a 10 GiB target HFile size and 100 pre-split regions: compute the stored size, the number of HFiles, and the reducer count.

**Hint.** Stored size is `input_gib * compression_ratio`. HFiles are `ceil(stored / target_hfile_gib)`, and set the number of reducers to the region count so each region receives one file per reducer.

**Expected result.** `800 GiB` stored, 80 HFiles, 100 reducers, and 100 target regions.

In [ ]:
import math

INPUT_GIB = 2_000
COMPRESSION_RATIO = 0.4
TARGET_HFILE_GIB = 10
PRE_SPLIT_REGIONS = 100

stored_gib = INPUT_GIB * COMPRESSION_RATIO
hfiles = math.ceil(stored_gib / TARGET_HFILE_GIB)
reducers = PRE_SPLIT_REGIONS
regions = PRE_SPLIT_REGIONS

print(f"stored size   = {stored_gib:.1f} GiB")
print(f"HFiles        = {hfiles} at about {TARGET_HFILE_GIB} GiB each")
print(f"reducers      = {reducers}")
print(f"target regions= {regions}")

assert stored_gib == 800.0
assert hfiles == 80
assert reducers == 100
assert regions == 100
assert hfiles / regions == 0.8
